# Notebook 04 — Nomination Optimization and Imbalance-Settlement Evaluation

This notebook converts the audited wind-scenario distributions from Notebook 03
into day-ahead nominations and evaluates their realised commercial settlement
performance for DK1 and DK2.

The notebook keeps the following objects separate:

- **forecast track:** D2 versus exact-run;
- **scenario stream:** raw versus gated recalibration;
- **nomination policy:** deterministic centre, fixed scenario quantiles,
  historical spread signal, and online utility-selected quantiles;
- **price area:** DK1 versus DK2.

No realised price or wind observation from the current delivery day is used to
choose that day's nomination policy.


## Commercial quantity

For actual production \(A_t\), day-ahead nomination \(N_t\), day-ahead price
\(P_t^{DA}\), and imbalance price \(P_t^{IMB}\),

\[
R_t
=
N_t P_t^{DA}
+
(A_t-N_t)P_t^{IMB}.
\]

Relative to perfect nomination \(A_tP_t^{DA}\),

\[
R_t-A_tP_t^{DA}
=
(A_t-N_t)(P_t^{IMB}-P_t^{DA}).
\]

A less negative—or more positive—incremental value is commercially better.

The wind target is hourly while Nordic settlement prices are quarter-hourly in
the evaluation period. The base analysis allocates the net hourly energy
deviation evenly across the four quarters, which is equivalent to multiplying
the hourly deviation by the mean quarter-hour price spread. This assumption is
audited explicitly.


## Optional dependencies

Run this only if a package is missing.

```python
# %pip install numpy pandas pyarrow scipy matplotlib requests
```


In [ ]:
# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable
import hashlib
import json
import math
import os
import platform
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from IPython.display import display
from requests.adapters import HTTPAdapter
from scipy import stats
from urllib3.util.retry import Retry


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

LOCAL_TIMEZONE = "Europe/Copenhagen"
TARGET_COLUMN = "actual_wind_mwh"

NOTEBOOK_03_OUTPUT_RUN = (
    "notebook_03_probabilistic"
)
NOTEBOOK_04_OUTPUT_RUN = (
    "notebook_04_commercial"
)

PRICE_AREAS = (
    "DK1",
    "DK2",
)

SCENARIO_STREAMS = (
    "raw",
    "gated",
)

# The historical market window seeds the price-spread signal.
MARKET_HISTORY_DAYS = 180

# Energi Data Service source transitions.
DAY_AHEAD_15MIN_TRANSITION_LOCAL_DATE = (
    pd.Timestamp("2025-10-01")
)
DAY_AHEAD_15MIN_TRANSITION_UTC = (
    DAY_AHEAD_15MIN_TRANSITION_LOCAL_DATE
    .tz_localize(LOCAL_TIMEZONE)
    .tz_convert("UTC")
)

IMBALANCE_15MIN_TRANSITION_LOCAL_DATE = (
    pd.Timestamp("2025-03-19")
)
IMBALANCE_15MIN_TRANSITION_UTC = (
    IMBALANCE_15MIN_TRANSITION_LOCAL_DATE
    .tz_localize(LOCAL_TIMEZONE)
    .tz_convert("UTC")
)

DAY_AHEAD_GATE_CLOSURE_LOCAL_HOUR = 12

# Energi Data Service settings.
EDS_API_BASE = (
    "https://api.energidataservice.dk/dataset"
)
REFRESH_MARKET_CACHE = False
STRICT_PRICE_COVERAGE = True
API_TIMEOUT_SECONDS = 90
REQUEST_PAUSE_SECONDS = 0.25
MAX_RECORDS_PER_MONTH = 20_000

# Scenario quantiles used for nomination policies.
QUANTILE_GRID = (
    0.10,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.75,
    0.80,
    0.90,
)

# Historical spread-signal policy.
SPREAD_SIGNAL_MAX_QUANTILE_SHIFT = 0.25
SPREAD_SIGNAL_SCALE_EUR_PER_MWH = 20.0
SPREAD_SIGNAL_HOUR_WEIGHT = 0.70
SPREAD_SIGNAL_MIN_HOUR_HISTORY = 10
SPREAD_SIGNAL_MIN_AREA_HISTORY = 48

# Online policy.
ONLINE_HISTORY_WINDOW_DAYS = 42
ONLINE_MINIMUM_HISTORY_DAYS = 14
ONLINE_COMPLETED_DAY_LAG = 2
ONLINE_DOWNSIDE_PROBABILITY = 0.20
ONLINE_RISK_WEIGHT = 0.25

# Commercial evaluation.
PORTFOLIO_SHARE = 1.0
BASE_ADDITIONAL_IMBALANCE_FEE_EUR_PER_MWH = 0.0
FEE_SENSITIVITY_EUR_PER_MWH = (
    0.0,
    0.25,
    0.50,
    1.00,
    2.00,
)
SPREAD_STRESS_MULTIPLIERS = (
    0.50,
    1.00,
    1.50,
    2.00,
)

# Time-series inference.
BLOCK_LENGTH_DAYS = 7
BOOTSTRAP_REPETITIONS = 5_000

print(
    {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "random_seed": RANDOM_SEED,
        "scenario_streams": SCENARIO_STREAMS,
        "quantile_grid": QUANTILE_GRID,
        "bootstrap_repetitions": (
            BOOTSTRAP_REPETITIONS
        ),
    }
)


In [ ]:
# ============================================================
# 2. PROJECT ROOT AND PATHS
# ============================================================

def find_project_root(
    starting_path: Path | None = None,
) -> Path:
    current = Path(
        starting_path or Path.cwd()
    ).resolve()

    for candidate in (
        current,
        *current.parents,
    ):
        if (
            candidate.joinpath(
                "pyproject.toml"
            ).exists()
            and candidate.joinpath(
                "data"
            ).exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected pyproject.toml and data/."
    )


PROJECT_ROOT = find_project_root()

NOTEBOOK_03_DIR = (
    PROJECT_ROOT
    / "outputs"
    / NOTEBOOK_03_OUTPUT_RUN
)

NOTEBOOK_03_AUDIT_DIR = (
    NOTEBOOK_03_DIR
    / "audits"
)

NOTEBOOK_03_PREDICTION_DIR = (
    NOTEBOOK_03_DIR
    / "predictions"
)

NOTEBOOK_03_SCENARIO_DIR = (
    NOTEBOOK_03_DIR
    / "scenarios"
)

NOTEBOOK_03_TABLE_DIR = (
    NOTEBOOK_03_DIR
    / "tables"
)

RAW_SCENARIO_PATH = (
    NOTEBOOK_03_SCENARIO_DIR
    / "notebook_03_joint_wind_scenarios.parquet"
)

GATED_SCENARIO_PATH = (
    NOTEBOOK_03_SCENARIO_DIR
    / (
        "notebook_03_joint_wind_scenarios_"
        "gated_recalibrated.parquet"
    )
)

NOTEBOOK_03_VERIFICATION_PATH = (
    NOTEBOOK_03_AUDIT_DIR
    / "notebook_03_verification.json"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / NOTEBOOK_04_OUTPUT_RUN
)

AUDIT_DIR = OUTPUT_DIR / "audits"
FIGURE_DIR = OUTPUT_DIR / "figures"
MARKET_DIR = OUTPUT_DIR / "market"
NOMINATION_DIR = OUTPUT_DIR / "nominations"
SETTLEMENT_DIR = OUTPUT_DIR / "settlement"
TABLE_DIR = OUTPUT_DIR / "tables"

RAW_MARKET_CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "market_prices_notebook_04"
)

for directory in (
    AUDIT_DIR,
    FIGURE_DIR,
    MARKET_DIR,
    NOMINATION_DIR,
    SETTLEMENT_DIR,
    TABLE_DIR,
    RAW_MARKET_CACHE_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Project root:", PROJECT_ROOT)
print("Notebook 03:", NOTEBOOK_03_DIR)
print("Notebook 04:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 3. VERIFY NOTEBOOK 03 AND LOAD SCENARIO STREAMS
# ============================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


for required_path in (
    NOTEBOOK_03_VERIFICATION_PATH,
    RAW_SCENARIO_PATH,
    GATED_SCENARIO_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required Notebook 03 artifact "
            f"not found: {required_path}"
        )


with NOTEBOOK_03_VERIFICATION_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook_03_verification = (
        json.load(file)
    )


if notebook_03_verification.get(
    "verification_status"
) != "PROBABILISTIC_MODELLING_COMPLETE":
    raise RuntimeError(
        "Notebook 03 is not verified as "
        "PROBABILISTIC_MODELLING_COMPLETE."
    )


raw_scenarios = pd.read_parquet(
    RAW_SCENARIO_PATH,
    columns=[
        "panel",
        "fold_id",
        "delivery_date_local",
        "timestamp_utc",
        "price_area",
        "scenario_id",
        "centre_model",
        "centre_prediction",
        TARGET_COLUMN,
        "scenario_wind_mwh",
    ],
)

raw_scenarios = raw_scenarios.rename(
    columns={
        "scenario_wind_mwh": (
            "scenario_wind_mwh_value"
        )
    }
)

raw_scenarios["scenario_stream"] = (
    "raw"
)


gated_scenarios = pd.read_parquet(
    GATED_SCENARIO_PATH,
    columns=[
        "panel",
        "fold_id",
        "delivery_date_local",
        "timestamp_utc",
        "price_area",
        "scenario_id",
        "centre_model",
        "centre_prediction",
        TARGET_COLUMN,
        "scenario_wind_mwh_gated",
    ],
)

gated_scenarios = gated_scenarios.rename(
    columns={
        "scenario_wind_mwh_gated": (
            "scenario_wind_mwh_value"
        )
    }
)

gated_scenarios["scenario_stream"] = (
    "gated"
)


all_scenarios = pd.concat(
    [
        raw_scenarios,
        gated_scenarios,
    ],
    ignore_index=True,
)


all_scenarios["timestamp_utc"] = (
    pd.to_datetime(
        all_scenarios[
            "timestamp_utc"
        ],
        utc=True,
    )
)

all_scenarios[
    "delivery_date_local"
] = pd.to_datetime(
    all_scenarios[
        "delivery_date_local"
    ]
)

all_scenarios[
    "fold_id"
] = pd.to_numeric(
    all_scenarios["fold_id"],
    errors="raise",
).astype(int)


required_columns = {
    "panel",
    "fold_id",
    "delivery_date_local",
    "timestamp_utc",
    "price_area",
    "scenario_id",
    "centre_model",
    "centre_prediction",
    TARGET_COLUMN,
    "scenario_wind_mwh_value",
    "scenario_stream",
}

missing_columns = (
    required_columns
    - set(all_scenarios.columns)
)

if missing_columns:
    raise RuntimeError(
        "Scenario inputs are missing columns: "
        f"{sorted(missing_columns)}"
    )


if (
    all_scenarios[
        [
            "centre_prediction",
            TARGET_COLUMN,
            "scenario_wind_mwh_value",
        ]
    ]
    .isna()
    .any()
    .any()
):
    raise RuntimeError(
        "Scenario inputs contain missing numeric values."
    )


if (
    all_scenarios[
        [
            "centre_prediction",
            TARGET_COLUMN,
            "scenario_wind_mwh_value",
        ]
    ]
    < -1e-10
).any().any():
    raise RuntimeError(
        "Negative wind values were found."
    )


scenario_key = [
    "panel",
    "fold_id",
    "timestamp_utc",
    "price_area",
    "scenario_stream",
    "scenario_id",
]

if all_scenarios.duplicated(
    scenario_key
).any():
    raise RuntimeError(
        "Duplicate scenario keys were found."
    )


scenario_counts = (
    all_scenarios.groupby(
        [
            "panel",
            "fold_id",
            "timestamp_utc",
            "price_area",
            "scenario_stream",
        ]
    )["scenario_id"]
    .nunique()
)

if scenario_counts.nunique() != 1:
    raise RuntimeError(
        "Scenario count is not constant across "
        "timestamps and streams."
    )

SCENARIO_COUNT = int(
    scenario_counts.iloc[0]
)


input_manifest = {
    "notebook_03_status": (
        notebook_03_verification.get(
            "verification_status"
        )
    ),
    "raw_scenario_path": str(
        RAW_SCENARIO_PATH
    ),
    "raw_scenario_sha256": sha256_file(
        RAW_SCENARIO_PATH
    ),
    "gated_scenario_path": str(
        GATED_SCENARIO_PATH
    ),
    "gated_scenario_sha256": sha256_file(
        GATED_SCENARIO_PATH
    ),
    "scenario_rows": int(
        len(all_scenarios)
    ),
    "scenario_count_per_timestamp": (
        SCENARIO_COUNT
    ),
}

print(
    json.dumps(
        input_manifest,
        indent=2,
    )
)


In [ ]:
# ============================================================
# 4. SCENARIO QUANTILES AND FORECAST METADATA
# ============================================================

scenario_group_columns = [
    "panel",
    "fold_id",
    "delivery_date_local",
    "timestamp_utc",
    "price_area",
    "centre_model",
    "scenario_stream",
]


metadata_consistency = (
    all_scenarios.groupby(
        scenario_group_columns
    )[
        [
            "centre_prediction",
            TARGET_COLUMN,
        ]
    ]
    .nunique(
        dropna=False
    )
)

if (
    metadata_consistency
    > 1
).any().any():
    raise RuntimeError(
        "Scenario members disagree on target or "
        "deterministic centre metadata."
    )


scenario_quantiles = (
    all_scenarios.groupby(
        scenario_group_columns
    )[
        "scenario_wind_mwh_value"
    ]
    .quantile(
        QUANTILE_GRID
    )
    .unstack()
    .reset_index()
)


def quantile_column(
    quantile: float,
) -> str:
    return (
        "q"
        + f"{quantile:.2f}".replace(
            ".",
            "_",
        )
    )


scenario_quantiles = (
    scenario_quantiles.rename(
        columns={
            quantile: (
                quantile_column(
                    quantile
                )
            )
            for quantile in (
                QUANTILE_GRID
            )
        }
    )
)


forecast_metadata = (
    all_scenarios[
        scenario_group_columns
        + [
            "centre_prediction",
            TARGET_COLUMN,
        ]
    ]
    .drop_duplicates(
        scenario_group_columns
    )
)


scenario_quantiles = (
    scenario_quantiles.merge(
        forecast_metadata,
        on=scenario_group_columns,
        how="left",
        validate="one_to_one",
    )
)


scenario_quantiles[
    "delivery_date"
] = (
    scenario_quantiles[
        "timestamp_utc"
    ]
    .dt.tz_convert(
        LOCAL_TIMEZONE
    )
    .dt.date
)

scenario_quantiles[
    "delivery_hour_local"
] = (
    scenario_quantiles[
        "timestamp_utc"
    ]
    .dt.tz_convert(
        LOCAL_TIMEZONE
    )
    .dt.hour
)


def gate_closure_utc(
    delivery_date: Any,
) -> pd.Timestamp:
    previous_local_date = (
        pd.Timestamp(
            delivery_date
        )
        - pd.Timedelta(
            days=1
        )
    )

    naive_gate = (
        previous_local_date
        + pd.Timedelta(
            hours=(
                DAY_AHEAD_GATE_CLOSURE_LOCAL_HOUR
            )
        )
    )

    return (
        naive_gate
        .tz_localize(
            LOCAL_TIMEZONE
        )
        .tz_convert("UTC")
    )


scenario_quantiles[
    "forecast_origin_utc"
] = scenario_quantiles[
    "delivery_date"
].map(
    gate_closure_utc
)


if (
    scenario_quantiles[
        "forecast_origin_utc"
    ]
    >= scenario_quantiles[
        "timestamp_utc"
    ]
).any():
    raise RuntimeError(
        "At least one forecast origin is not before "
        "delivery."
    )


EVALUATION_START_UTC = (
    scenario_quantiles[
        "timestamp_utc"
    ]
    .min()
    .floor("D")
)

EVALUATION_END_UTC = (
    scenario_quantiles[
        "timestamp_utc"
    ]
    .max()
    .floor("D")
    + pd.Timedelta(
        days=1
    )
)

MARKET_HISTORY_START_UTC = (
    EVALUATION_START_UTC
    - pd.Timedelta(
        days=MARKET_HISTORY_DAYS
    )
)

print(
    {
        "scenario_quantile_rows": int(
            len(scenario_quantiles)
        ),
        "evaluation_start_utc": (
            EVALUATION_START_UTC
        ),
        "evaluation_end_utc": (
            EVALUATION_END_UTC
        ),
        "market_history_start_utc": (
            MARKET_HISTORY_START_UTC
        ),
    }
)


## Market-price data

The notebook downloads and caches:

- `Elspotprices` for hourly day-ahead prices before 1 October 2025;
- `DayAheadPrices` for quarter-hour day-ahead prices from that date;
- `ImbalancePrice` for quarter-hour imbalance prices.

The exact source transition is applied after schema normalization.


In [ ]:
# ============================================================
# 5. ENERGI DATA SERVICE HELPERS
# ============================================================

def stable_hash(
    payload: Any,
) -> str:
    serialised = json.dumps(
        payload,
        sort_keys=True,
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(
        serialised
    ).hexdigest()[:12]


def build_http_session() -> requests.Session:
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=frozenset(
            ["GET"]
        ),
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry
    )

    session = requests.Session()
    session.mount(
        "https://",
        adapter,
    )
    session.mount(
        "http://",
        adapter,
    )

    session.headers.update(
        {
            "User-Agent": (
                "Nordic-Wind-Decision-System-"
                "Notebook-04/1.0"
            )
        }
    )

    return session


def monthly_intervals(
    start_utc: pd.Timestamp,
    end_utc: pd.Timestamp,
) -> Iterable[
    tuple[
        pd.Timestamp,
        pd.Timestamp,
    ]
]:
    if (
        start_utc.tzinfo is None
        or end_utc.tzinfo is None
    ):
        raise ValueError(
            "monthly_intervals requires "
            "timezone-aware timestamps."
        )

    cursor = pd.Timestamp(
        year=start_utc.year,
        month=start_utc.month,
        day=1,
        tz="UTC",
    )

    while cursor < end_utc:
        next_cursor = (
            cursor
            + pd.offsets.MonthBegin(1)
        )

        yield (
            max(
                cursor,
                start_utc,
            ),
            min(
                next_cursor,
                end_utc,
            ),
        )

        cursor = next_cursor


def fetch_eds_dataset_monthly(
    dataset: str,
    start_utc: pd.Timestamp,
    end_utc: pd.Timestamp,
    price_areas: tuple[str, ...],
    cache_path: Path,
    *,
    refresh: bool = False,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    request_log_path = (
        cache_path.with_suffix(
            ".requests.csv"
        )
    )

    if (
        cache_path.exists()
        and not refresh
    ):
        cached = pd.read_parquet(
            cache_path
        )

        request_log = (
            pd.read_csv(
                request_log_path
            )
            if request_log_path.exists()
            else pd.DataFrame()
        )

        print(
            f"Loaded cached {dataset}: "
            f"{len(cached):,} rows"
        )

        return (
            cached,
            request_log,
        )

    session = build_http_session()

    all_records: list[
        pd.DataFrame
    ] = []

    request_rows: list[
        dict[str, Any]
    ] = []

    for (
        chunk_start,
        chunk_end,
    ) in monthly_intervals(
        start_utc,
        end_utc,
    ):
        params = {
            "start": (
                chunk_start.strftime(
                    "%Y-%m-%d"
                )
            ),
            "end": (
                chunk_end.strftime(
                    "%Y-%m-%d"
                )
            ),
            "limit": (
                MAX_RECORDS_PER_MONTH
            ),
            "filter": json.dumps(
                {
                    "PriceArea": (
                        list(
                            price_areas
                        )
                    )
                },
                separators=(
                    ",",
                    ":",
                ),
            ),
        }

        url = (
            f"{EDS_API_BASE}/"
            f"{dataset}"
        )

        response = session.get(
            url,
            params=params,
            timeout=(
                API_TIMEOUT_SECONDS
            ),
        )

        if not response.ok:
            raise RuntimeError(
                "Energi Data Service request "
                f"failed for {dataset}: "
                f"HTTP {response.status_code}\n"
                f"{response.text[:1000]}\n"
                f"Request URL: {response.url}"
            )

        payload = response.json()
        records = payload.get(
            "records"
        )

        if records is None:
            raise KeyError(
                "Energi Data Service response "
                f"for {dataset} has no records. "
                f"Keys: {sorted(payload)}"
            )

        total = int(
            payload.get(
                "total",
                len(records),
            )
        )

        if total > len(records):
            raise RuntimeError(
                f"Monthly request for {dataset} "
                "was truncated: "
                f"total={total}, "
                f"returned={len(records)}."
            )

        frame = pd.DataFrame.from_records(
            records
        )

        frame[
            "source_dataset"
        ] = dataset

        frame[
            "request_start_utc"
        ] = chunk_start

        frame[
            "request_end_utc"
        ] = chunk_end

        all_records.append(
            frame
        )

        request_rows.append(
            {
                "dataset": dataset,
                "chunk_start_utc": (
                    chunk_start
                ),
                "chunk_end_utc": (
                    chunk_end
                ),
                "http_status": (
                    response.status_code
                ),
                "records": len(
                    records
                ),
                "reported_total": total,
                "request_url": (
                    response.url
                ),
            }
        )

        time.sleep(
            REQUEST_PAUSE_SECONDS
        )

    if not all_records:
        raise RuntimeError(
            f"No records returned for "
            f"{dataset}."
        )

    result = pd.concat(
        all_records,
        ignore_index=True,
        sort=False,
    )

    request_log = pd.DataFrame(
        request_rows
    )

    result.to_parquet(
        cache_path,
        index=False,
    )

    request_log.to_csv(
        request_log_path,
        index=False,
    )

    print(
        f"Downloaded {dataset}: "
        f"{len(result):,} rows"
    )

    return (
        result,
        request_log,
    )


def resolve_column(
    frame: pd.DataFrame,
    candidates: list[str],
    semantic_name: str,
) -> str:
    def canonical(
        value: str,
    ) -> str:
        return "".join(
            character.lower()
            for character in str(value)
            if character.isalnum()
        )

    canonical_map = {
        canonical(column): column
        for column in frame.columns
    }

    for candidate in candidates:
        key = canonical(
            candidate
        )

        if key in canonical_map:
            return canonical_map[
                key
            ]

    raise KeyError(
        f"Could not resolve {semantic_name}. "
        f"Candidates={candidates}. "
        f"Available={list(frame.columns)}"
    )


In [ ]:
# ============================================================
# 6. DOWNLOAD OR LOAD MARKET PRICES
# ============================================================

market_signature = stable_hash(
    {
        "start": (
            MARKET_HISTORY_START_UTC
        ),
        "end": (
            EVALUATION_END_UTC
        ),
        "areas": PRICE_AREAS,
        "day_ahead_transition": (
            DAY_AHEAD_15MIN_TRANSITION_UTC
        ),
    }
)

request_logs = []


legacy_da_raw = pd.DataFrame()

if (
    MARKET_HISTORY_START_UTC
    < DAY_AHEAD_15MIN_TRANSITION_UTC
):
    legacy_query_end = min(
        EVALUATION_END_UTC,
        DAY_AHEAD_15MIN_TRANSITION_UTC.ceil(
            "D"
        ),
    )

    legacy_da_raw, request_log = (
        fetch_eds_dataset_monthly(
            dataset="Elspotprices",
            start_utc=(
                MARKET_HISTORY_START_UTC
            ),
            end_utc=(
                legacy_query_end
            ),
            price_areas=(
                PRICE_AREAS
            ),
            cache_path=(
                RAW_MARKET_CACHE_DIR
                / (
                    "elspotprices_"
                    f"{market_signature}.parquet"
                )
            ),
            refresh=(
                REFRESH_MARKET_CACHE
            ),
        )
    )

    request_logs.append(
        request_log
    )


new_da_raw = pd.DataFrame()

if (
    EVALUATION_END_UTC
    > DAY_AHEAD_15MIN_TRANSITION_UTC
):
    new_query_start = max(
        MARKET_HISTORY_START_UTC,
        DAY_AHEAD_15MIN_TRANSITION_UTC.floor(
            "D"
        ),
    )

    new_da_raw, request_log = (
        fetch_eds_dataset_monthly(
            dataset="DayAheadPrices",
            start_utc=(
                new_query_start
            ),
            end_utc=(
                EVALUATION_END_UTC
            ),
            price_areas=(
                PRICE_AREAS
            ),
            cache_path=(
                RAW_MARKET_CACHE_DIR
                / (
                    "dayaheadprices_"
                    f"{market_signature}.parquet"
                )
            ),
            refresh=(
                REFRESH_MARKET_CACHE
            ),
        )
    )

    request_logs.append(
        request_log
    )


imbalance_raw, request_log = (
    fetch_eds_dataset_monthly(
        dataset="ImbalancePrice",
        start_utc=(
            MARKET_HISTORY_START_UTC
        ),
        end_utc=(
            EVALUATION_END_UTC
        ),
        price_areas=PRICE_AREAS,
        cache_path=(
            RAW_MARKET_CACHE_DIR
            / (
                "imbalanceprice_"
                f"{market_signature}.parquet"
            )
        ),
        refresh=(
            REFRESH_MARKET_CACHE
        ),
    )
)

request_logs.append(
    request_log
)


market_request_log = pd.concat(
    [
        frame
        for frame in request_logs
        if not frame.empty
    ],
    ignore_index=True,
)

display(
    market_request_log
)


In [ ]:
# ============================================================
# 7. NORMALIZE MARKET SCHEMAS
# ============================================================

TIMESTAMP_CANDIDATES = [
    "TimeUTC",
    "HourUTC",
    "Minutes5UTC",
    "QuarterUTC",
    "MTUStartUTC",
    "MtuStartUTC",
    "IntervalStartUTC",
    "PeriodStartUTC",
]

AREA_CANDIDATES = [
    "PriceArea",
    "Area",
    "BiddingZone",
]

DAY_AHEAD_EUR_CANDIDATES = [
    "DayAheadPriceEUR",
    "DayAheadPriceEur",
    "SpotPriceEUR",
    "SpotPriceEur",
    "PriceEUR",
    "PriceEur",
]

IMBALANCE_EUR_CANDIDATES = [
    "ImbalancePriceEUR",
    "ImbalancePriceEur",
    "PriceEUR",
    "PriceEur",
]


def normalize_price_frame(
    raw: pd.DataFrame,
    price_candidates: list[str],
    output_price_column: str,
    source_name: str,
) -> pd.DataFrame:
    if raw.empty:
        return pd.DataFrame(
            columns=[
                "interval_start_utc",
                "price_area",
                output_price_column,
                "source_dataset",
            ]
        )

    timestamp_column = resolve_column(
        raw,
        TIMESTAMP_CANDIDATES,
        (
            f"{source_name} "
            "UTC timestamp"
        ),
    )

    area_column = resolve_column(
        raw,
        AREA_CANDIDATES,
        (
            f"{source_name} "
            "price area"
        ),
    )

    price_column = resolve_column(
        raw,
        price_candidates,
        (
            f"{source_name} "
            "EUR price"
        ),
    )

    result = raw[
        [
            timestamp_column,
            area_column,
            price_column,
        ]
    ].copy()

    result = result.rename(
        columns={
            timestamp_column: (
                "interval_start_utc"
            ),
            area_column: (
                "price_area"
            ),
            price_column: (
                output_price_column
            ),
        }
    )

    result[
        "interval_start_utc"
    ] = pd.to_datetime(
        result[
            "interval_start_utc"
        ],
        utc=True,
        errors="raise",
    )

    result["price_area"] = (
        result["price_area"]
        .astype(str)
    )

    result[
        output_price_column
    ] = pd.to_numeric(
        result[
            output_price_column
        ],
        errors="coerce",
    )

    result[
        "source_dataset"
    ] = source_name

    result = result.loc[
        result[
            "price_area"
        ].isin(
            PRICE_AREAS
        )
        & result[
            "interval_start_utc"
        ].ge(
            MARKET_HISTORY_START_UTC
        )
        & result[
            "interval_start_utc"
        ].lt(
            EVALUATION_END_UTC
        )
    ].copy()

    if not np.isfinite(
        result[
            output_price_column
        ].to_numpy(
            dtype=float
        )
    ).all():
        raise RuntimeError(
            f"{source_name} contains "
            "missing or non-finite prices."
        )

    key = [
        "interval_start_utc",
        "price_area",
    ]

    duplicate_mask = (
        result.duplicated(
            key,
            keep=False,
        )
    )

    if duplicate_mask.any():
        conflicting = (
            result.loc[
                duplicate_mask
            ]
            .groupby(key)[
                output_price_column
            ]
            .nunique(
                dropna=False
            )
        )

        if (
            conflicting
            > 1
        ).any():
            raise RuntimeError(
                f"{source_name} contains "
                "conflicting duplicate prices."
            )

        result = (
            result.drop_duplicates(
                key,
                keep="last",
            )
        )

    return (
        result.sort_values(
            key
        )
        .reset_index(
            drop=True
        )
    )


legacy_da = normalize_price_frame(
    legacy_da_raw,
    DAY_AHEAD_EUR_CANDIDATES,
    "day_ahead_price_eur_per_mwh",
    "Elspotprices",
)

new_da = normalize_price_frame(
    new_da_raw,
    DAY_AHEAD_EUR_CANDIDATES,
    "day_ahead_price_eur_per_mwh",
    "DayAheadPrices",
)

imbalance_quarter = (
    normalize_price_frame(
        imbalance_raw,
        IMBALANCE_EUR_CANDIDATES,
        "imbalance_price_eur_per_mwh",
        "ImbalancePrice",
    )
)


legacy_da = legacy_da.loc[
    legacy_da[
        "interval_start_utc"
    ].lt(
        DAY_AHEAD_15MIN_TRANSITION_UTC
    )
].copy()

new_da = new_da.loc[
    new_da[
        "interval_start_utc"
    ].ge(
        DAY_AHEAD_15MIN_TRANSITION_UTC
    )
].copy()


if not legacy_da.empty:
    if not legacy_da[
        "interval_start_utc"
    ].dt.minute.eq(0).all():
        raise RuntimeError(
            "Legacy day-ahead prices "
            "are not hourly."
        )


if not new_da.empty:
    if not new_da[
        "interval_start_utc"
    ].dt.minute.isin(
        [
            0,
            15,
            30,
            45,
        ]
    ).all():
        raise RuntimeError(
            "New day-ahead prices are "
            "not quarter-hourly."
        )


if not imbalance_quarter[
    "interval_start_utc"
].dt.minute.isin(
    [
        0,
        15,
        30,
        45,
    ]
).all():
    raise RuntimeError(
        "Imbalance prices are not "
        "quarter-hourly in the requested "
        "period."
    )


print(
    {
        "legacy_day_ahead_rows": (
            len(legacy_da)
        ),
        "new_day_ahead_rows": (
            len(new_da)
        ),
        "imbalance_rows": (
            len(imbalance_quarter)
        ),
    }
)


In [ ]:
# ============================================================
# 8. BUILD QUARTER-HOUR AND HOURLY MARKET TABLES
# ============================================================

imbalance_quarter = (
    imbalance_quarter.copy()
)

imbalance_quarter[
    "hour_utc"
] = (
    imbalance_quarter[
        "interval_start_utc"
    ].dt.floor("h")
)


legacy_da_hour = (
    legacy_da.rename(
        columns={
            "interval_start_utc": (
                "hour_utc"
            )
        }
    )[
        [
            "hour_utc",
            "price_area",
            "day_ahead_price_eur_per_mwh",
            "source_dataset",
        ]
    ]
    .rename(
        columns={
            "source_dataset": (
                "day_ahead_source"
            )
        }
    )
)


pre_transition_market = (
    imbalance_quarter.loc[
        imbalance_quarter[
            "interval_start_utc"
        ].lt(
            DAY_AHEAD_15MIN_TRANSITION_UTC
        )
    ]
    .merge(
        legacy_da_hour,
        on=[
            "hour_utc",
            "price_area",
        ],
        how="left",
        validate="many_to_one",
    )
)


new_da_quarter = (
    new_da[
        [
            "interval_start_utc",
            "price_area",
            "day_ahead_price_eur_per_mwh",
            "source_dataset",
        ]
    ]
    .rename(
        columns={
            "source_dataset": (
                "day_ahead_source"
            )
        }
    )
)


post_transition_market = (
    imbalance_quarter.loc[
        imbalance_quarter[
            "interval_start_utc"
        ].ge(
            DAY_AHEAD_15MIN_TRANSITION_UTC
        )
    ]
    .merge(
        new_da_quarter,
        on=[
            "interval_start_utc",
            "price_area",
        ],
        how="left",
        validate="one_to_one",
    )
)


market_quarter = pd.concat(
    [
        pre_transition_market,
        post_transition_market,
    ],
    ignore_index=True,
    sort=False,
)


market_quarter[
    "spread_eur_per_mwh"
] = (
    market_quarter[
        "imbalance_price_eur_per_mwh"
    ]
    - market_quarter[
        "day_ahead_price_eur_per_mwh"
    ]
)


market_quarter[
    "market_design_regime"
] = np.where(
    market_quarter[
        "interval_start_utc"
    ].lt(
        DAY_AHEAD_15MIN_TRANSITION_UTC
    ),
    (
        "hourly_DA_plus_"
        "15min_imbalance"
    ),
    (
        "15min_DA_and_imbalance"
    ),
)


quarter_key = [
    "interval_start_utc",
    "price_area",
]

quarter_duplicates = int(
    market_quarter.duplicated(
        quarter_key
    ).sum()
)

if quarter_duplicates:
    raise RuntimeError(
        "Duplicate quarter-hour market "
        f"keys: {quarter_duplicates}"
    )


missing_quarter_prices = (
    market_quarter[
        [
            "day_ahead_price_eur_per_mwh",
            "imbalance_price_eur_per_mwh",
        ]
    ]
    .isna()
    .any(axis=1)
)

if missing_quarter_prices.any():
    missing = market_quarter.loc[
        missing_quarter_prices,
        [
            "interval_start_utc",
            "price_area",
            "market_design_regime",
        ],
    ]

    display(
        missing.head(100)
    )

    if STRICT_PRICE_COVERAGE:
        raise RuntimeError(
            "Missing market prices for "
            f"{len(missing)} quarter-hours."
        )


market_quarter = (
    market_quarter.loc[
        ~missing_quarter_prices
    ].copy()
)


market_hourly = (
    market_quarter.groupby(
        [
            "hour_utc",
            "price_area",
            "market_design_regime",
        ],
        as_index=False,
    )
    .agg(
        quarter_count=(
            "interval_start_utc",
            "nunique",
        ),
        day_ahead_price_eur_per_mwh=(
            "day_ahead_price_eur_per_mwh",
            "mean",
        ),
        imbalance_price_eur_per_mwh=(
            "imbalance_price_eur_per_mwh",
            "mean",
        ),
        spread_mean_eur_per_mwh=(
            "spread_eur_per_mwh",
            "mean",
        ),
        spread_min_eur_per_mwh=(
            "spread_eur_per_mwh",
            "min",
        ),
        spread_max_eur_per_mwh=(
            "spread_eur_per_mwh",
            "max",
        ),
        spread_std_eur_per_mwh=(
            "spread_eur_per_mwh",
            "std",
        ),
    )
    .rename(
        columns={
            "hour_utc": (
                "timestamp_utc"
            )
        }
    )
)


market_hourly[
    "spread_std_eur_per_mwh"
] = market_hourly[
    "spread_std_eur_per_mwh"
].fillna(0.0)


incomplete_hours = (
    market_hourly.loc[
        ~market_hourly[
            "quarter_count"
        ].eq(4)
    ].copy()
)

if not incomplete_hours.empty:
    display(
        incomplete_hours.head(100)
    )

    if STRICT_PRICE_COVERAGE:
        raise RuntimeError(
            "Found market hours without "
            "exactly four quarter-hours: "
            f"{len(incomplete_hours)}"
        )


if market_hourly.duplicated(
    [
        "timestamp_utc",
        "price_area",
    ]
).any():
    raise RuntimeError(
        "Duplicate hourly market keys."
    )


market_hourly[
    "market_available_utc"
] = (
    market_hourly[
        "timestamp_utc"
    ]
    + pd.Timedelta(
        hours=1
    )
)

local_market_timestamp = (
    market_hourly[
        "timestamp_utc"
    ]
    .dt.tz_convert(
        LOCAL_TIMEZONE
    )
)

market_hourly[
    "delivery_hour_local"
] = (
    local_market_timestamp.dt.hour
)

market_hourly[
    "delivery_date"
] = (
    local_market_timestamp.dt.date
)


market_quarter.to_parquet(
    MARKET_DIR
    / "market_prices_quarter_hour.parquet",
    index=False,
)

market_hourly.to_parquet(
    MARKET_DIR
    / "market_prices_hourly.parquet",
    index=False,
)

market_request_log.to_csv(
    AUDIT_DIR
    / "market_api_request_log.csv",
    index=False,
)

print(
    {
        "quarter_rows": (
            len(market_quarter)
        ),
        "hour_rows": (
            len(market_hourly)
        ),
        "incomplete_hours": (
            len(incomplete_hours)
        ),
    }
)


## Information-safe price-spread signal

The spread signal uses only completed historical market hours available before
the day-ahead forecast origin. It blends:

- an exponentially weighted mean for the same price area and local hour;
- an area-wide exponentially weighted mean.

A positive expected spread lowers the nomination quantile; a negative expected
spread raises it.


In [ ]:
# ============================================================
# 9. BUILD INFORMATION-SAFE MARKET-SPREAD SIGNALS
# ============================================================

market_signal_source = (
    market_hourly.sort_values(
        [
            "price_area",
            "delivery_hour_local",
            "market_available_utc",
        ]
    )
    .copy()
)


market_signal_source[
    "hour_spread_ewm"
] = (
    market_signal_source.groupby(
        [
            "price_area",
            "delivery_hour_local",
        ]
    )[
        "spread_mean_eur_per_mwh"
    ]
    .transform(
        lambda values: values.ewm(
            halflife=14,
            adjust=False,
            min_periods=1,
        ).mean()
    )
)


market_signal_source[
    "hour_history_count"
] = (
    market_signal_source.groupby(
        [
            "price_area",
            "delivery_hour_local",
        ]
    )
    .cumcount()
    + 1
)


area_signal_source = (
    market_hourly.sort_values(
        [
            "price_area",
            "market_available_utc",
        ]
    )
    .copy()
)


area_signal_source[
    "area_spread_ewm"
] = (
    area_signal_source.groupby(
        "price_area"
    )[
        "spread_mean_eur_per_mwh"
    ]
    .transform(
        lambda values: values.ewm(
            halflife=14 * 24,
            adjust=False,
            min_periods=1,
        ).mean()
    )
)


area_signal_source[
    "area_history_count"
] = (
    area_signal_source.groupby(
        "price_area"
    )
    .cumcount()
    + 1
)


left = (
    scenario_quantiles.sort_values(
        [
            "forecast_origin_utc",
            "price_area",
            "delivery_hour_local",
        ]
    )
    .copy()
)


hour_signal_table = (
    market_signal_source[
        [
            "market_available_utc",
            "price_area",
            "delivery_hour_local",
            "hour_spread_ewm",
            "hour_history_count",
        ]
    ]
    .sort_values(
        [
            "market_available_utc",
            "price_area",
            "delivery_hour_local",
        ]
    )
)


signal_enriched = pd.merge_asof(
    left,
    hour_signal_table,
    left_on="forecast_origin_utc",
    right_on="market_available_utc",
    by=[
        "price_area",
        "delivery_hour_local",
    ],
    direction="backward",
    allow_exact_matches=True,
)


signal_enriched = (
    signal_enriched.rename(
        columns={
            "market_available_utc": (
                "hour_signal_latest_market_utc"
            )
        }
    )
)


area_signal_table = (
    area_signal_source[
        [
            "market_available_utc",
            "price_area",
            "area_spread_ewm",
            "area_history_count",
        ]
    ]
    .sort_values(
        [
            "market_available_utc",
            "price_area",
        ]
    )
)


signal_enriched = pd.merge_asof(
    signal_enriched.sort_values(
        [
            "forecast_origin_utc",
            "price_area",
        ]
    ),
    area_signal_table,
    left_on="forecast_origin_utc",
    right_on="market_available_utc",
    by="price_area",
    direction="backward",
    allow_exact_matches=True,
)


signal_enriched = (
    signal_enriched.rename(
        columns={
            "market_available_utc": (
                "area_signal_latest_market_utc"
            )
        }
    )
)


hour_is_usable = (
    signal_enriched[
        "hour_history_count"
    ].fillna(0)
    >= SPREAD_SIGNAL_MIN_HOUR_HISTORY
)

area_is_usable = (
    signal_enriched[
        "area_history_count"
    ].fillna(0)
    >= SPREAD_SIGNAL_MIN_AREA_HISTORY
)


signal_enriched[
    "expected_spread_eur_per_mwh"
] = np.where(
    hour_is_usable
    & area_is_usable,
    (
        SPREAD_SIGNAL_HOUR_WEIGHT
        * signal_enriched[
            "hour_spread_ewm"
        ]
        + (
            1.0
            - SPREAD_SIGNAL_HOUR_WEIGHT
        )
        * signal_enriched[
            "area_spread_ewm"
        ]
    ),
    np.where(
        hour_is_usable,
        signal_enriched[
            "hour_spread_ewm"
        ],
        np.where(
            area_is_usable,
            signal_enriched[
                "area_spread_ewm"
            ],
            0.0,
        ),
    ),
)


signal_enriched[
    "continuous_spread_signal_quantile"
] = (
    0.50
    - SPREAD_SIGNAL_MAX_QUANTILE_SHIFT
    * np.tanh(
        signal_enriched[
            "expected_spread_eur_per_mwh"
        ]
        / SPREAD_SIGNAL_SCALE_EUR_PER_MWH
    )
)


quantile_grid_array = np.asarray(
    QUANTILE_GRID,
    dtype=float,
)


def nearest_quantile(
    value: float,
) -> float:
    return float(
        quantile_grid_array[
            int(
                np.argmin(
                    np.abs(
                        quantile_grid_array
                        - float(value)
                    )
                )
            )
        ]
    )


signal_enriched[
    "spread_signal_quantile"
] = signal_enriched[
    "continuous_spread_signal_quantile"
].map(
    nearest_quantile
)


signal_lookahead_violations = int(
    (
        signal_enriched[
            "hour_signal_latest_market_utc"
        ].notna()
        & (
            signal_enriched[
                "hour_signal_latest_market_utc"
            ]
            > signal_enriched[
                "forecast_origin_utc"
            ]
        )
    ).sum()
    + (
        signal_enriched[
            "area_signal_latest_market_utc"
        ].notna()
        & (
            signal_enriched[
                "area_signal_latest_market_utc"
            ]
            > signal_enriched[
                "forecast_origin_utc"
            ]
        )
    ).sum()
)


if signal_lookahead_violations:
    raise RuntimeError(
        "Market-spread signal uses future "
        f"market data in "
        f"{signal_lookahead_violations} rows."
    )


display(
    signal_enriched.groupby(
        [
            "panel",
            "price_area",
            "scenario_stream",
        ],
        as_index=False,
    )
    .agg(
        mean_expected_spread=(
            "expected_spread_eur_per_mwh",
            "mean",
        ),
        mean_signal_quantile=(
            "spread_signal_quantile",
            "mean",
        ),
        minimum_hour_history=(
            "hour_history_count",
            "min",
        ),
        minimum_area_history=(
            "area_history_count",
            "min",
        ),
    )
)


In [ ]:
# ============================================================
# 10. MERGE EVALUATION MARKET PRICES
# ============================================================

evaluation_market = (
    market_hourly.loc[
        market_hourly[
            "timestamp_utc"
        ].ge(
            EVALUATION_START_UTC
        )
        & market_hourly[
            "timestamp_utc"
        ].lt(
            EVALUATION_END_UTC
        )
    ]
    .copy()
)


quantile_market = (
    signal_enriched.merge(
        evaluation_market[
            [
                "timestamp_utc",
                "price_area",
                "quarter_count",
                "day_ahead_price_eur_per_mwh",
                "imbalance_price_eur_per_mwh",
                "spread_mean_eur_per_mwh",
                "spread_min_eur_per_mwh",
                "spread_max_eur_per_mwh",
                "spread_std_eur_per_mwh",
                "market_design_regime",
            ]
        ],
        on=[
            "timestamp_utc",
            "price_area",
        ],
        how="left",
        validate="many_to_one",
    )
)


price_columns = [
    "day_ahead_price_eur_per_mwh",
    "imbalance_price_eur_per_mwh",
    "spread_mean_eur_per_mwh",
    "spread_min_eur_per_mwh",
    "spread_max_eur_per_mwh",
]


missing_market_rows = (
    quantile_market[
        price_columns
    ]
    .isna()
    .any(axis=1)
)


if missing_market_rows.any():
    missing_keys = (
        quantile_market.loc[
            missing_market_rows,
            [
                "timestamp_utc",
                "price_area",
            ],
        ]
        .drop_duplicates()
        .sort_values(
            [
                "price_area",
                "timestamp_utc",
            ]
        )
    )

    display(
        missing_keys.head(100)
    )

    if STRICT_PRICE_COVERAGE:
        raise RuntimeError(
            "Missing complete market prices for "
            f"{len(missing_keys)} forecast keys."
        )


quantile_market = (
    quantile_market.loc[
        ~missing_market_rows
    ].copy()
)


if not quantile_market[
    "quarter_count"
].eq(4).all():
    raise RuntimeError(
        "At least one evaluation hour does not "
        "contain four market quarters."
    )


price_coverage_report = (
    signal_enriched[
        [
            "timestamp_utc",
            "price_area",
        ]
    ]
    .drop_duplicates()
    .merge(
        evaluation_market[
            [
                "timestamp_utc",
                "price_area",
                "quarter_count",
            ]
        ],
        on=[
            "timestamp_utc",
            "price_area",
        ],
        how="left",
        validate="one_to_one",
    )
)


price_coverage_report[
    "complete_market_hour"
] = (
    price_coverage_report[
        "quarter_count"
    ].eq(4)
)


price_coverage_report.to_csv(
    AUDIT_DIR
    / "market_price_coverage.csv",
    index=False,
)

print(
    {
        "quantile_market_rows": (
            len(quantile_market)
        ),
        "missing_market_rows": (
            int(
                missing_market_rows.sum()
            )
        ),
    }
)


## Nomination policies

The notebook evaluates six deployable policies:

1. deterministic centre;
2. fixed scenario 25th percentile;
3. scenario median;
4. fixed scenario 75th percentile;
5. historical spread-signal quantile;
6. online profit-selected quantile;
7. online risk-adjusted quantile.

The online policies select one scenario quantile per delivery day using only
completed days through \(D-2\). Their warm-up fallback is the information-safe
spread-signal quantile.


In [ ]:
# ============================================================
# 11. BUILD FIXED AND SPREAD-SIGNAL NOMINATIONS
# ============================================================

def values_from_selected_quantile(
    frame: pd.DataFrame,
    selected_quantiles: pd.Series,
) -> np.ndarray:
    column_lookup = {
        quantile: quantile_column(
            quantile
        )
        for quantile in QUANTILE_GRID
    }

    output = np.empty(
        len(frame),
        dtype=float,
    )

    selected_values = (
        selected_quantiles
        .to_numpy(dtype=float)
    )

    for quantile in QUANTILE_GRID:
        mask = np.isclose(
            selected_values,
            quantile,
        )

        if mask.any():
            output[mask] = frame.loc[
                mask,
                column_lookup[
                    quantile
                ],
            ].to_numpy(
                dtype=float
            )

    return output


base_nomination_columns = [
    "panel",
    "fold_id",
    "delivery_date",
    "delivery_date_local",
    "timestamp_utc",
    "forecast_origin_utc",
    "delivery_hour_local",
    "price_area",
    "centre_model",
    "scenario_stream",
    TARGET_COLUMN,
    "centre_prediction",
    "expected_spread_eur_per_mwh",
    "spread_signal_quantile",
    "day_ahead_price_eur_per_mwh",
    "imbalance_price_eur_per_mwh",
    "spread_mean_eur_per_mwh",
    "spread_min_eur_per_mwh",
    "spread_max_eur_per_mwh",
    "spread_std_eur_per_mwh",
    "market_design_regime",
]


fixed_nomination_specs = {
    "point_centre": None,
    "scenario_q25": 0.25,
    "scenario_median": 0.50,
    "scenario_q75": 0.75,
}


nomination_parts = []


for (
    method_name,
    quantile,
) in fixed_nomination_specs.items():
    block = quantile_market[
        base_nomination_columns
    ].copy()

    block[
        "nomination_method"
    ] = method_name

    if quantile is None:
        block[
            "selected_quantile"
        ] = np.nan

        block[
            "nominated_mwh"
        ] = block[
            "centre_prediction"
        ]
    else:
        block[
            "selected_quantile"
        ] = quantile

        block[
            "nominated_mwh"
        ] = quantile_market[
            quantile_column(
                quantile
            )
        ].to_numpy(
            dtype=float
        )

    block[
        "online_history_days"
    ] = np.nan

    block[
        "online_fallback_used"
    ] = False

    nomination_parts.append(
        block
    )


spread_signal_block = (
    quantile_market[
        base_nomination_columns
    ].copy()
)

spread_signal_block[
    "nomination_method"
] = (
    "spread_signal_quantile"
)

spread_signal_block[
    "selected_quantile"
] = (
    quantile_market[
        "spread_signal_quantile"
    ].to_numpy(
        dtype=float
    )
)

spread_signal_block[
    "nominated_mwh"
] = (
    values_from_selected_quantile(
        quantile_market,
        quantile_market[
            "spread_signal_quantile"
        ],
    )
)

spread_signal_block[
    "online_history_days"
] = np.nan

spread_signal_block[
    "online_fallback_used"
] = False

nomination_parts.append(
    spread_signal_block
)


In [ ]:
# ============================================================
# 12. CANDIDATE QUANTILE UTILITY HISTORY
# ============================================================

candidate_daily_parts = []


for quantile in QUANTILE_GRID:
    candidate = quantile_market[
        [
            "panel",
            "fold_id",
            "delivery_date",
            "price_area",
            "centre_model",
            "scenario_stream",
            TARGET_COLUMN,
            "spread_mean_eur_per_mwh",
        ]
    ].copy()

    candidate[
        "candidate_quantile"
    ] = quantile

    candidate[
        "candidate_nomination_mwh"
    ] = quantile_market[
        quantile_column(
            quantile
        )
    ].to_numpy(
        dtype=float
    )

    candidate[
        "candidate_imbalance_mwh"
    ] = (
        candidate[
            TARGET_COLUMN
        ]
        - candidate[
            "candidate_nomination_mwh"
        ]
    )

    candidate[
        "candidate_incremental_eur"
    ] = (
        candidate[
            "candidate_imbalance_mwh"
        ]
        * candidate[
            "spread_mean_eur_per_mwh"
        ]
    )

    candidate[
        "candidate_absolute_imbalance_mwh"
    ] = candidate[
        "candidate_imbalance_mwh"
    ].abs()

    daily_candidate = (
        candidate.groupby(
            [
                "panel",
                "fold_id",
                "delivery_date",
                "price_area",
                "centre_model",
                "scenario_stream",
                "candidate_quantile",
            ],
            as_index=False,
        )
        .agg(
            daily_incremental_eur=(
                "candidate_incremental_eur",
                "sum",
            ),
            daily_actual_mwh=(
                TARGET_COLUMN,
                "sum",
            ),
            daily_absolute_imbalance_mwh=(
                "candidate_absolute_imbalance_mwh",
                "sum",
            ),
        )
    )

    daily_candidate[
        "daily_incremental_eur_per_actual_mwh"
    ] = (
        daily_candidate[
            "daily_incremental_eur"
        ]
        / daily_candidate[
            "daily_actual_mwh"
        ].replace(
            0.0,
            np.nan,
        )
    )

    candidate_daily_parts.append(
        daily_candidate
    )


candidate_daily_utility = pd.concat(
    candidate_daily_parts,
    ignore_index=True,
)


candidate_daily_utility.to_parquet(
    NOMINATION_DIR
    / "candidate_quantile_daily_utility.parquet",
    index=False,
)


print(
    {
        "candidate_daily_rows": (
            len(
                candidate_daily_utility
            )
        ),
        "delivery_days": int(
            candidate_daily_utility[
                "delivery_date"
            ].nunique()
        ),
    }
)


In [ ]:
# ============================================================
# 13. INFORMATION-SAFE ONLINE QUANTILE SELECTION
# ============================================================

def downside_cvar(
    values: np.ndarray,
    probability: float,
) -> float:
    array = np.asarray(
        values,
        dtype=float,
    )

    array = array[
        np.isfinite(array)
    ]

    if len(array) == 0:
        return np.nan

    threshold = np.quantile(
        array,
        probability,
    )

    tail = array[
        array <= threshold
    ]

    return float(
        tail.mean()
        if len(tail)
        else threshold
    )


def choose_quantile_from_scores(
    scores: pd.Series,
) -> float:
    maximum = float(
        scores.max()
    )

    candidates = (
        scores.loc[
            np.isclose(
                scores,
                maximum,
            )
        ]
        .index
        .to_numpy(dtype=float)
    )

    return float(
        candidates[
            int(
                np.argmin(
                    np.abs(
                        candidates
                        - 0.50
                    )
                )
            )
        ]
    )


daily_spread_fallback = (
    quantile_market.groupby(
        [
            "panel",
            "fold_id",
            "delivery_date",
            "price_area",
            "centre_model",
            "scenario_stream",
        ],
        as_index=False,
    )
    .agg(
        fallback_quantile=(
            "spread_signal_quantile",
            "median",
        )
    )
)


online_decision_rows = []
online_lookahead_violations = 0


decision_group_columns = [
    "panel",
    "price_area",
    "centre_model",
    "scenario_stream",
]


for (
    panel_name,
    price_area,
    centre_model,
    scenario_stream,
), group in candidate_daily_utility.groupby(
    decision_group_columns,
    sort=True,
):
    utility_wide = (
        group.pivot_table(
            index=[
                "fold_id",
                "delivery_date",
            ],
            columns=(
                "candidate_quantile"
            ),
            values=(
                "daily_incremental_eur_per_actual_mwh"
            ),
            aggfunc="first",
        )
        .sort_index(
            level=[
                "delivery_date",
                "fold_id",
            ]
        )
    )

    date_rows = (
        utility_wide.reset_index()[
            [
                "fold_id",
                "delivery_date",
            ]
        ]
        .sort_values(
            [
                "delivery_date",
                "fold_id",
            ]
        )
        .drop_duplicates()
    )

    fallback_lookup = (
        daily_spread_fallback.loc[
            daily_spread_fallback[
                "panel"
            ].eq(panel_name)
            & daily_spread_fallback[
                "price_area"
            ].eq(price_area)
            & daily_spread_fallback[
                "centre_model"
            ].eq(centre_model)
            & daily_spread_fallback[
                "scenario_stream"
            ].eq(scenario_stream)
        ]
        .set_index(
            [
                "fold_id",
                "delivery_date",
            ]
        )[
            "fallback_quantile"
        ]
        .to_dict()
    )

    for row in date_rows.itertuples(
        index=False
    ):
        fold_id = int(
            row.fold_id
        )
        delivery_date = (
            row.delivery_date
        )

        latest_safe_date = (
            delivery_date
            - pd.Timedelta(
                days=(
                    ONLINE_COMPLETED_DAY_LAG
                )
            ).to_pytimedelta()
        )

        earliest_safe_date = (
            latest_safe_date
            - pd.Timedelta(
                days=(
                    ONLINE_HISTORY_WINDOW_DAYS
                    - 1
                )
            ).to_pytimedelta()
        )

        history = group.loc[
            group[
                "delivery_date"
            ].ge(
                earliest_safe_date
            )
            & group[
                "delivery_date"
            ].le(
                latest_safe_date
            )
        ].copy()

        history_days = int(
            history[
                "delivery_date"
            ].nunique()
        )

        latest_history_date = (
            history[
                "delivery_date"
            ].max()
            if not history.empty
            else None
        )

        if (
            latest_history_date
            is not None
            and latest_history_date
            > latest_safe_date
        ):
            online_lookahead_violations += 1

        fallback_quantile = float(
            fallback_lookup[
                (
                    fold_id,
                    delivery_date,
                )
            ]
        )

        if (
            history_days
            < ONLINE_MINIMUM_HISTORY_DAYS
        ):
            profit_quantile = (
                fallback_quantile
            )

            risk_quantile = (
                fallback_quantile
            )

            fallback_used = True
        else:
            mean_score = (
                history.groupby(
                    "candidate_quantile"
                )[
                    "daily_incremental_eur_per_actual_mwh"
                ]
                .mean()
            )

            profit_quantile = (
                choose_quantile_from_scores(
                    mean_score
                )
            )

            risk_score_rows = {}

            for (
                candidate_quantile,
                candidate_group,
            ) in history.groupby(
                "candidate_quantile"
            ):
                values = candidate_group[
                    "daily_incremental_eur_per_actual_mwh"
                ].to_numpy(
                    dtype=float
                )

                mean_value = float(
                    np.mean(values)
                )

                cvar_value = downside_cvar(
                    values,
                    ONLINE_DOWNSIDE_PROBABILITY,
                )

                risk_score_rows[
                    float(
                        candidate_quantile
                    )
                ] = (
                    mean_value
                    + ONLINE_RISK_WEIGHT
                    * cvar_value
                )

            risk_score = pd.Series(
                risk_score_rows
            )

            risk_quantile = (
                choose_quantile_from_scores(
                    risk_score
                )
            )

            fallback_used = False

        online_decision_rows.append(
            {
                "panel": panel_name,
                "fold_id": fold_id,
                "delivery_date": (
                    delivery_date
                ),
                "price_area": price_area,
                "centre_model": (
                    centre_model
                ),
                "scenario_stream": (
                    scenario_stream
                ),
                "history_days": (
                    history_days
                ),
                "latest_history_date": (
                    latest_history_date
                ),
                "latest_safe_date": (
                    latest_safe_date
                ),
                "fallback_quantile": (
                    fallback_quantile
                ),
                "online_profit_quantile": (
                    profit_quantile
                ),
                "online_risk_adjusted_quantile": (
                    risk_quantile
                ),
                "fallback_used": (
                    fallback_used
                ),
            }
        )


online_quantile_decisions = pd.DataFrame(
    online_decision_rows
)


if online_lookahead_violations:
    raise RuntimeError(
        "Online quantile selection contains "
        f"{online_lookahead_violations} "
        "look-ahead violations."
    )


online_quantile_decisions.to_csv(
    AUDIT_DIR
    / "online_quantile_decisions.csv",
    index=False,
)


display(
    online_quantile_decisions.groupby(
        [
            "panel",
            "price_area",
            "scenario_stream",
        ],
        as_index=False,
    )
    .agg(
        mean_profit_quantile=(
            "online_profit_quantile",
            "mean",
        ),
        mean_risk_quantile=(
            "online_risk_adjusted_quantile",
            "mean",
        ),
        fallback_rate=(
            "fallback_used",
            "mean",
        ),
        maximum_history_days=(
            "history_days",
            "max",
        ),
    )
)


In [ ]:
# ============================================================
# 14. BUILD ONLINE NOMINATIONS
# ============================================================

online_source = (
    quantile_market.merge(
        online_quantile_decisions,
        on=[
            "panel",
            "fold_id",
            "delivery_date",
            "price_area",
            "centre_model",
            "scenario_stream",
        ],
        how="left",
        validate="many_to_one",
    )
)


if online_source[
    [
        "online_profit_quantile",
        "online_risk_adjusted_quantile",
    ]
].isna().any().any():
    raise RuntimeError(
        "Online quantile decisions are missing."
    )


for (
    method_name,
    decision_column,
) in (
    (
        "online_profit_quantile",
        "online_profit_quantile",
    ),
    (
        "online_risk_adjusted_quantile",
        "online_risk_adjusted_quantile",
    ),
):
    block = online_source[
        base_nomination_columns
    ].copy()

    block[
        "nomination_method"
    ] = method_name

    block[
        "selected_quantile"
    ] = online_source[
        decision_column
    ].to_numpy(
        dtype=float
    )

    block[
        "nominated_mwh"
    ] = values_from_selected_quantile(
        online_source,
        online_source[
            decision_column
        ],
    )

    block[
        "online_history_days"
    ] = online_source[
        "history_days"
    ].to_numpy(
        dtype=float
    )

    block[
        "online_fallback_used"
    ] = online_source[
        "fallback_used"
    ].to_numpy(
        dtype=bool
    )

    nomination_parts.append(
        block
    )


nominations = pd.concat(
    nomination_parts,
    ignore_index=True,
)


nominations[
    "nominated_mwh"
] = np.maximum(
    nominations[
        "nominated_mwh"
    ],
    0.0,
)


nomination_key = [
    "panel",
    "fold_id",
    "timestamp_utc",
    "price_area",
    "scenario_stream",
    "nomination_method",
]


if nominations.duplicated(
    nomination_key
).any():
    raise RuntimeError(
        "Duplicate nomination keys were found."
    )


expected_methods = {
    "point_centre",
    "scenario_q25",
    "scenario_median",
    "scenario_q75",
    "spread_signal_quantile",
    "online_profit_quantile",
    "online_risk_adjusted_quantile",
}


if set(
    nominations[
        "nomination_method"
    ].unique()
) != expected_methods:
    raise RuntimeError(
        "Nomination method set is incomplete."
    )


nominations.to_parquet(
    NOMINATION_DIR
    / "notebook_04_nominations_hourly.parquet",
    index=False,
)


display(
    nominations.groupby(
        [
            "panel",
            "price_area",
            "scenario_stream",
            "nomination_method",
        ],
        as_index=False,
    )
    .agg(
        hours=(
            "timestamp_utc",
            "size",
        ),
        mean_nomination_mwh=(
            "nominated_mwh",
            "mean",
        ),
        mean_selected_quantile=(
            "selected_quantile",
            "mean",
        ),
    )
)


In [ ]:
# ============================================================
# 15. SETTLEMENT QUANTITIES
# ============================================================

commercial = nominations.copy()


commercial[
    "actual_mwh"
] = (
    commercial[
        TARGET_COLUMN
    ]
    * PORTFOLIO_SHARE
)


commercial[
    "nominated_mwh"
] = (
    commercial[
        "nominated_mwh"
    ]
    * PORTFOLIO_SHARE
)


commercial[
    "imbalance_mwh"
] = (
    commercial[
        "actual_mwh"
    ]
    - commercial[
        "nominated_mwh"
    ]
)


commercial[
    "absolute_imbalance_mwh"
] = commercial[
    "imbalance_mwh"
].abs()


commercial[
    "day_ahead_revenue_eur"
] = (
    commercial[
        "nominated_mwh"
    ]
    * commercial[
        "day_ahead_price_eur_per_mwh"
    ]
)


commercial[
    "imbalance_settlement_eur"
] = (
    commercial[
        "imbalance_mwh"
    ]
    * commercial[
        "imbalance_price_eur_per_mwh"
    ]
)


commercial[
    "gross_revenue_eur"
] = (
    commercial[
        "day_ahead_revenue_eur"
    ]
    + commercial[
        "imbalance_settlement_eur"
    ]
)


commercial[
    "perfect_nomination_revenue_eur"
] = (
    commercial[
        "actual_mwh"
    ]
    * commercial[
        "day_ahead_price_eur_per_mwh"
    ]
)


commercial[
    "incremental_revenue_vs_perfect_eur"
] = (
    commercial[
        "imbalance_mwh"
    ]
    * commercial[
        "spread_mean_eur_per_mwh"
    ]
)


commercial[
    "additional_imbalance_fee_eur"
] = (
    BASE_ADDITIONAL_IMBALANCE_FEE_EUR_PER_MWH
    * commercial[
        "absolute_imbalance_mwh"
    ]
)


commercial[
    "net_incremental_revenue_vs_perfect_eur"
] = (
    commercial[
        "incremental_revenue_vs_perfect_eur"
    ]
    - commercial[
        "additional_imbalance_fee_eur"
    ]
)


bound_at_min = (
    commercial[
        "imbalance_mwh"
    ]
    * commercial[
        "spread_min_eur_per_mwh"
    ]
)


bound_at_max = (
    commercial[
        "imbalance_mwh"
    ]
    * commercial[
        "spread_max_eur_per_mwh"
    ]
)


commercial[
    "incremental_revenue_same_sign_lower_eur"
] = np.minimum(
    bound_at_min,
    bound_at_max,
)


commercial[
    "incremental_revenue_same_sign_upper_eur"
] = np.maximum(
    bound_at_min,
    bound_at_max,
)


revenue_identity_error = (
    commercial[
        "gross_revenue_eur"
    ]
    - commercial[
        "perfect_nomination_revenue_eur"
    ]
    - commercial[
        "incremental_revenue_vs_perfect_eur"
    ]
).abs()


maximum_revenue_identity_error = float(
    revenue_identity_error.max()
)


if (
    maximum_revenue_identity_error
    > 1e-7
):
    raise RuntimeError(
        "Commercial revenue identity failed: "
        f"{maximum_revenue_identity_error}"
    )


support_group = [
    "panel",
    "fold_id",
    "price_area",
    "scenario_stream",
]


support_mismatches = []


for keys, group in commercial.groupby(
    support_group,
    sort=True,
):
    reference = set(
        group.loc[
            group[
                "nomination_method"
            ].eq(
                "point_centre"
            ),
            "timestamp_utc",
        ]
    )

    for (
        method,
        method_group,
    ) in group.groupby(
        "nomination_method"
    ):
        observed = set(
            method_group[
                "timestamp_utc"
            ]
        )

        if observed != reference:
            support_mismatches.append(
                {
                    "context": keys,
                    "method": method,
                    "missing": len(
                        reference
                        - observed
                    ),
                    "extra": len(
                        observed
                        - reference
                    ),
                }
            )


if support_mismatches:
    raise RuntimeError(
        "Nomination policies do not share "
        "common timestamp support: "
        f"{support_mismatches[:10]}"
    )


commercial.to_parquet(
    SETTLEMENT_DIR
    / "notebook_04_settlement_hourly.parquet",
    index=False,
)


print(
    {
        "commercial_rows": int(
            len(commercial)
        ),
        "maximum_revenue_identity_error": (
            maximum_revenue_identity_error
        ),
        "support_mismatches": (
            len(support_mismatches)
        ),
    }
)


In [ ]:
# ============================================================
# 16. COMMERCIAL PERFORMANCE TABLES
# ============================================================

def commercial_metrics(
    group: pd.DataFrame,
) -> dict[str, Any]:
    actual = group[
        "actual_mwh"
    ].to_numpy(
        dtype=float
    )

    imbalance = group[
        "imbalance_mwh"
    ].to_numpy(
        dtype=float
    )

    incremental = group[
        "net_incremental_revenue_vs_perfect_eur"
    ].to_numpy(
        dtype=float
    )

    daily = (
        group.groupby(
            "delivery_date",
            as_index=False,
        )[
            "net_incremental_revenue_vs_perfect_eur"
        ]
        .sum()
    )

    total_actual = float(
        actual.sum()
    )

    gross_revenue = float(
        group[
            "gross_revenue_eur"
        ].sum()
    )

    perfect_revenue = float(
        group[
            "perfect_nomination_revenue_eur"
        ].sum()
    )

    total_incremental = float(
        incremental.sum()
    )

    return {
        "hours": int(
            len(group)
        ),
        "days": int(
            group[
                "delivery_date"
            ].nunique()
        ),
        "folds": int(
            group[
                "fold_id"
            ].nunique()
        ),
        "total_actual_mwh": (
            total_actual
        ),
        "total_nominated_mwh": float(
            group[
                "nominated_mwh"
            ].sum()
        ),
        "total_absolute_imbalance_mwh": float(
            np.abs(
                imbalance
            ).sum()
        ),
        "mean_absolute_imbalance_mwh": float(
            np.abs(
                imbalance
            ).mean()
        ),
        "mean_signed_imbalance_mwh": float(
            imbalance.mean()
        ),
        "gross_revenue_eur": (
            gross_revenue
        ),
        "perfect_nomination_revenue_eur": (
            perfect_revenue
        ),
        "total_incremental_revenue_vs_perfect_eur": (
            total_incremental
        ),
        "settlement_shortfall_eur_per_actual_mwh": (
            -total_incremental
            / total_actual
            if total_actual > 0
            else np.nan
        ),
        "incremental_eur_per_actual_mwh": (
            total_incremental
            / total_actual
            if total_actual > 0
            else np.nan
        ),
        "gross_revenue_capture_pct": (
            100.0
            * gross_revenue
            / perfect_revenue
            if abs(
                perfect_revenue
            )
            > 1e-12
            else np.nan
        ),
        "mean_hourly_incremental_eur": float(
            incremental.mean()
        ),
        "median_hourly_incremental_eur": float(
            np.median(
                incremental
            )
        ),
        "positive_incremental_hours_pct": float(
            100.0
            * np.mean(
                incremental > 0
            )
        ),
        "hourly_downside_cvar_5pct_eur": (
            downside_cvar(
                incremental,
                0.05,
            )
        ),
        "worst_hour_incremental_eur": float(
            incremental.min()
        ),
        "worst_day_incremental_eur": float(
            daily[
                "net_incremental_revenue_vs_perfect_eur"
            ].min()
        ),
        "mean_within_hour_spread_std_eur_per_mwh": float(
            group[
                "spread_std_eur_per_mwh"
            ].mean()
        ),
    }


commercial_summary_rows = []


for keys, group in commercial.groupby(
    [
        "panel",
        "price_area",
        "scenario_stream",
        "nomination_method",
    ],
    sort=True,
):
    (
        panel_name,
        price_area,
        scenario_stream,
        nomination_method,
    ) = keys

    commercial_summary_rows.append(
        {
            "panel": panel_name,
            "price_area": price_area,
            "scenario_stream": (
                scenario_stream
            ),
            "nomination_method": (
                nomination_method
            ),
            **commercial_metrics(
                group
            ),
        }
    )


commercial_summary = (
    pd.DataFrame(
        commercial_summary_rows
    )
    .sort_values(
        [
            "panel",
            "price_area",
            "scenario_stream",
            "total_incremental_revenue_vs_perfect_eur",
        ],
        ascending=[
            True,
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


commercial_fold_summary_rows = []


for keys, group in commercial.groupby(
    [
        "panel",
        "fold_id",
        "price_area",
        "scenario_stream",
        "nomination_method",
    ],
    sort=True,
):
    (
        panel_name,
        fold_id,
        price_area,
        scenario_stream,
        nomination_method,
    ) = keys

    commercial_fold_summary_rows.append(
        {
            "panel": panel_name,
            "fold_id": fold_id,
            "price_area": price_area,
            "scenario_stream": (
                scenario_stream
            ),
            "nomination_method": (
                nomination_method
            ),
            **commercial_metrics(
                group
            ),
        }
    )


commercial_fold_summary = (
    pd.DataFrame(
        commercial_fold_summary_rows
    )
)


display(
    commercial_summary[
        [
            "panel",
            "price_area",
            "scenario_stream",
            "nomination_method",
            "total_incremental_revenue_vs_perfect_eur",
            "incremental_eur_per_actual_mwh",
            "total_absolute_imbalance_mwh",
            "gross_revenue_capture_pct",
            "hourly_downside_cvar_5pct_eur",
        ]
    ]
)


In [ ]:
# ============================================================
# 17. PAIRED COMPARISON AGAINST THE POINT NOMINATION
# ============================================================

point_reference = (
    commercial.loc[
        commercial[
            "nomination_method"
        ].eq(
            "point_centre"
        ),
        [
            "panel",
            "fold_id",
            "delivery_date",
            "timestamp_utc",
            "price_area",
            "scenario_stream",
            "net_incremental_revenue_vs_perfect_eur",
            "absolute_imbalance_mwh",
        ],
    ]
    .rename(
        columns={
            "net_incremental_revenue_vs_perfect_eur": (
                "point_incremental_eur"
            ),
            "absolute_imbalance_mwh": (
                "point_absolute_imbalance_mwh"
            ),
        }
    )
)


comparison_vs_point = (
    commercial.loc[
        ~commercial[
            "nomination_method"
        ].eq(
            "point_centre"
        )
    ]
    .merge(
        point_reference,
        on=[
            "panel",
            "fold_id",
            "delivery_date",
            "timestamp_utc",
            "price_area",
            "scenario_stream",
        ],
        how="inner",
        validate="many_to_one",
    )
)


comparison_vs_point[
    "commercial_delta_vs_point_eur"
] = (
    comparison_vs_point[
        "net_incremental_revenue_vs_perfect_eur"
    ]
    - comparison_vs_point[
        "point_incremental_eur"
    ]
)


comparison_vs_point[
    "absolute_imbalance_delta_vs_point_mwh"
] = (
    comparison_vs_point[
        "absolute_imbalance_mwh"
    ]
    - comparison_vs_point[
        "point_absolute_imbalance_mwh"
    ]
)


paired_rows = []


for keys, group in comparison_vs_point.groupby(
    [
        "panel",
        "price_area",
        "scenario_stream",
        "nomination_method",
    ],
    sort=True,
):
    (
        panel_name,
        price_area,
        scenario_stream,
        nomination_method,
    ) = keys

    fold_totals = (
        group.groupby(
            "fold_id",
            as_index=False,
        )[
            "commercial_delta_vs_point_eur"
        ]
        .sum()
    )

    paired_rows.append(
        {
            "panel": panel_name,
            "price_area": price_area,
            "scenario_stream": (
                scenario_stream
            ),
            "nomination_method": (
                nomination_method
            ),
            "hours": int(
                len(group)
            ),
            "total_commercial_delta_vs_point_eur": float(
                group[
                    "commercial_delta_vs_point_eur"
                ].sum()
            ),
            "mean_hourly_commercial_delta_vs_point_eur": float(
                group[
                    "commercial_delta_vs_point_eur"
                ].mean()
            ),
            "total_absolute_imbalance_delta_vs_point_mwh": float(
                group[
                    "absolute_imbalance_delta_vs_point_mwh"
                ].sum()
            ),
            "folds_won_vs_point": int(
                (
                    fold_totals[
                        "commercial_delta_vs_point_eur"
                    ]
                    > 0
                ).sum()
            ),
            "folds_evaluated": int(
                len(
                    fold_totals
                )
            ),
        }
    )


paired_vs_point = (
    pd.DataFrame(
        paired_rows
    )
    .sort_values(
        [
            "panel",
            "price_area",
            "scenario_stream",
            "total_commercial_delta_vs_point_eur",
        ],
        ascending=[
            True,
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


display(
    paired_vs_point
)


In [ ]:
# ============================================================
# 18. TIME-SERIES-AWARE INFERENCE
# ============================================================

def circular_block_sample(
    values: np.ndarray,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:
    array = np.asarray(
        values,
        dtype=float,
    )

    n = len(array)

    if n == 0:
        return array.copy()

    sampled = []

    while len(sampled) < n:
        start = int(
            rng.integers(
                0,
                n,
            )
        )

        indices = (
            start
            + np.arange(
                block_length
            )
        ) % n

        sampled.extend(
            array[
                indices
            ].tolist()
        )

    return np.asarray(
        sampled[:n],
        dtype=float,
    )


def within_fold_block_bootstrap_means(
    daily_frame: pd.DataFrame,
    value_column: str,
    *,
    repetitions: int,
    block_length: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(
        seed
    )

    fold_arrays = [
        group.sort_values(
            "delivery_date"
        )[
            value_column
        ].to_numpy(
            dtype=float
        )
        for _, group in daily_frame.groupby(
            "fold_id",
            sort=True,
        )
    ]

    result = np.empty(
        repetitions,
        dtype=float,
    )

    for repetition in range(
        repetitions
    ):
        sampled_parts = [
            circular_block_sample(
                values,
                block_length,
                rng,
            )
            for values in (
                fold_arrays
            )
        ]

        result[
            repetition
        ] = float(
            np.concatenate(
                sampled_parts
            ).mean()
        )

    return result


def hac_mean_inference(
    daily_frame: pd.DataFrame,
    value_column: str,
) -> dict[str, float]:
    ordered = daily_frame.sort_values(
        [
            "fold_id",
            "delivery_date",
        ]
    )

    values = ordered[
        value_column
    ].to_numpy(
        dtype=float
    )

    n = len(values)

    if n < 2:
        raise ValueError(
            "Inference requires at least "
            "two daily observations."
        )

    mean_value = float(
        values.mean()
    )

    long_run_sum = 0.0

    for _, fold_group in ordered.groupby(
        "fold_id",
        sort=True,
    ):
        fold_values = fold_group[
            value_column
        ].to_numpy(
            dtype=float
        )

        residuals = (
            fold_values
            - mean_value
        )

        fold_n = len(
            residuals
        )

        long_run_sum += float(
            np.dot(
                residuals,
                residuals,
            )
        )

        maximum_lag = min(
            BLOCK_LENGTH_DAYS,
            fold_n - 1,
        )

        for lag in range(
            1,
            maximum_lag + 1,
        ):
            weight = (
                1.0
                - lag
                / (
                    BLOCK_LENGTH_DAYS
                    + 1.0
                )
            )

            covariance_sum = float(
                np.dot(
                    residuals[
                        lag:
                    ],
                    residuals[
                        :-lag
                    ],
                )
            )

            long_run_sum += (
                2.0
                * weight
                * covariance_sum
            )

    variance = max(
        long_run_sum
        / (n ** 2),
        0.0,
    )

    standard_error = float(
        np.sqrt(
            variance
        )
    )

    t_statistic = (
        mean_value
        / standard_error
        if standard_error > 0
        else np.nan
    )

    p_value = (
        float(
            2.0
            * stats.norm.sf(
                abs(
                    t_statistic
                )
            )
        )
        if np.isfinite(
            t_statistic
        )
        else np.nan
    )

    return {
        "mean_daily_delta_eur": (
            mean_value
        ),
        "hac_lags": (
            BLOCK_LENGTH_DAYS
        ),
        "hac_standard_error": (
            standard_error
        ),
        "hac_t_statistic": (
            t_statistic
        ),
        "hac_p_value_two_sided": (
            p_value
        ),
    }


def holm_adjust(
    p_values: pd.Series,
) -> pd.Series:
    p_values = p_values.astype(
        float
    )

    order = np.argsort(
        p_values.to_numpy()
    )

    ordered = (
        p_values.to_numpy()[
            order
        ]
    )

    adjusted_ordered = np.empty_like(
        ordered
    )

    running_maximum = 0.0
    m = len(
        ordered
    )

    for rank, value in enumerate(
        ordered
    ):
        adjusted = min(
            1.0,
            (
                m - rank
            )
            * value,
        )

        running_maximum = max(
            running_maximum,
            adjusted,
        )

        adjusted_ordered[
            rank
        ] = running_maximum

    adjusted = np.empty_like(
        adjusted_ordered
    )

    adjusted[
        order
    ] = adjusted_ordered

    return pd.Series(
        adjusted,
        index=p_values.index,
    )


inference_rows = []


for keys, group in comparison_vs_point.groupby(
    [
        "panel",
        "price_area",
        "scenario_stream",
        "nomination_method",
    ],
    sort=True,
):
    (
        panel_name,
        price_area,
        scenario_stream,
        nomination_method,
    ) = keys

    daily = (
        group.groupby(
            [
                "fold_id",
                "delivery_date",
            ],
            as_index=False,
        )[
            "commercial_delta_vs_point_eur"
        ]
        .sum()
    )

    seed_payload = (
        f"{RANDOM_SEED}|"
        f"{panel_name}|"
        f"{price_area}|"
        f"{scenario_stream}|"
        f"{nomination_method}"
    )

    seed = int(
        hashlib.sha256(
            seed_payload.encode()
        ).hexdigest()[:8],
        16,
    )

    bootstrap = (
        within_fold_block_bootstrap_means(
            daily,
            "commercial_delta_vs_point_eur",
            repetitions=(
                BOOTSTRAP_REPETITIONS
            ),
            block_length=(
                BLOCK_LENGTH_DAYS
            ),
            seed=seed,
        )
    )

    probability = float(
        np.mean(
            bootstrap > 0
        )
    )

    bootstrap_p_value = float(
        min(
            1.0,
            2.0
            * min(
                probability,
                1.0
                - probability,
            ),
        )
    )

    inference_rows.append(
        {
            "panel": panel_name,
            "price_area": price_area,
            "scenario_stream": (
                scenario_stream
            ),
            "nomination_method": (
                nomination_method
            ),
            "days": int(
                len(daily)
            ),
            "folds": int(
                daily[
                    "fold_id"
                ].nunique()
            ),
            "bootstrap_design": (
                "7_day_circular_blocks_"
                "within_fold"
            ),
            "bootstrap_ci_lower_mean_daily_eur": float(
                np.quantile(
                    bootstrap,
                    0.025,
                )
            ),
            "bootstrap_ci_upper_mean_daily_eur": float(
                np.quantile(
                    bootstrap,
                    0.975,
                )
            ),
            "bootstrap_probability_outperformance": (
                probability
            ),
            "bootstrap_p_value_two_sided": (
                bootstrap_p_value
            ),
            **hac_mean_inference(
                daily,
                "commercial_delta_vs_point_eur",
            ),
        }
    )


inference_vs_point = pd.DataFrame(
    inference_rows
)


inference_vs_point[
    "holm_hac_p_value"
] = (
    inference_vs_point.groupby(
        [
            "panel",
            "price_area",
            "scenario_stream",
        ],
        group_keys=False,
    )[
        "hac_p_value_two_sided"
    ]
    .apply(
        holm_adjust
    )
)


inference_vs_point[
    "holm_bootstrap_p_value"
] = (
    inference_vs_point.groupby(
        [
            "panel",
            "price_area",
            "scenario_stream",
        ],
        group_keys=False,
    )[
        "bootstrap_p_value_two_sided"
    ]
    .apply(
        holm_adjust
    )
)


inference_vs_point = (
    inference_vs_point.sort_values(
        [
            "panel",
            "price_area",
            "scenario_stream",
            "mean_daily_delta_eur",
        ],
        ascending=[
            True,
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


display(
    inference_vs_point
)


In [ ]:
# ============================================================
# 19. RAW VERSUS GATED COMMERCIAL COMPARISON
# ============================================================

stream_daily = (
    commercial.groupby(
        [
            "panel",
            "fold_id",
            "delivery_date",
            "price_area",
            "nomination_method",
            "scenario_stream",
        ],
        as_index=False,
    )
    .agg(
        daily_incremental_eur=(
            "net_incremental_revenue_vs_perfect_eur",
            "sum",
        ),
        daily_absolute_imbalance_mwh=(
            "absolute_imbalance_mwh",
            "sum",
        ),
    )
)


stream_wide = (
    stream_daily.pivot(
        index=[
            "panel",
            "fold_id",
            "delivery_date",
            "price_area",
            "nomination_method",
        ],
        columns="scenario_stream",
        values=[
            "daily_incremental_eur",
            "daily_absolute_imbalance_mwh",
        ],
    )
)


stream_wide.columns = [
    f"{metric}_{stream}"
    for (
        metric,
        stream,
    ) in stream_wide.columns
]


stream_wide = (
    stream_wide.reset_index()
)


required_stream_columns = {
    "daily_incremental_eur_raw",
    "daily_incremental_eur_gated",
    "daily_absolute_imbalance_mwh_raw",
    "daily_absolute_imbalance_mwh_gated",
}


if not required_stream_columns.issubset(
    stream_wide.columns
):
    raise RuntimeError(
        "Raw-versus-gated comparison is incomplete."
    )


stream_wide[
    "gated_delta_vs_raw_eur"
] = (
    stream_wide[
        "daily_incremental_eur_gated"
    ]
    - stream_wide[
        "daily_incremental_eur_raw"
    ]
)


stream_wide[
    "gated_absolute_imbalance_delta_mwh"
] = (
    stream_wide[
        "daily_absolute_imbalance_mwh_gated"
    ]
    - stream_wide[
        "daily_absolute_imbalance_mwh_raw"
    ]
)


stream_comparison_rows = []


for keys, group in stream_wide.groupby(
    [
        "panel",
        "price_area",
        "nomination_method",
    ],
    sort=True,
):
    (
        panel_name,
        price_area,
        nomination_method,
    ) = keys

    bootstrap = (
        within_fold_block_bootstrap_means(
            group[
                [
                    "fold_id",
                    "delivery_date",
                    "gated_delta_vs_raw_eur",
                ]
            ],
            "gated_delta_vs_raw_eur",
            repetitions=(
                BOOTSTRAP_REPETITIONS
            ),
            block_length=(
                BLOCK_LENGTH_DAYS
            ),
            seed=int(
                hashlib.sha256(
                    (
                        f"{RANDOM_SEED}|"
                        f"{panel_name}|"
                        f"{price_area}|"
                        f"{nomination_method}|"
                        "stream"
                    ).encode()
                ).hexdigest()[:8],
                16,
            ),
        )
    )

    probability = float(
        np.mean(
            bootstrap > 0
        )
    )

    stream_comparison_rows.append(
        {
            "panel": panel_name,
            "price_area": price_area,
            "nomination_method": (
                nomination_method
            ),
            "days": int(
                len(group)
            ),
            "folds": int(
                group[
                    "fold_id"
                ].nunique()
            ),
            "total_gated_delta_vs_raw_eur": float(
                group[
                    "gated_delta_vs_raw_eur"
                ].sum()
            ),
            "mean_daily_gated_delta_vs_raw_eur": float(
                group[
                    "gated_delta_vs_raw_eur"
                ].mean()
            ),
            "total_absolute_imbalance_delta_mwh": float(
                group[
                    "gated_absolute_imbalance_delta_mwh"
                ].sum()
            ),
            "bootstrap_ci_lower_mean_daily_eur": float(
                np.quantile(
                    bootstrap,
                    0.025,
                )
            ),
            "bootstrap_ci_upper_mean_daily_eur": float(
                np.quantile(
                    bootstrap,
                    0.975,
                )
            ),
            "bootstrap_probability_gated_outperformance": (
                probability
            ),
        }
    )


raw_vs_gated_summary = (
    pd.DataFrame(
        stream_comparison_rows
    )
    .sort_values(
        [
            "panel",
            "price_area",
            "total_gated_delta_vs_raw_eur",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)


display(
    raw_vs_gated_summary
)


In [ ]:
# ============================================================
# 20. DENMARK PORTFOLIO RESULTS
# ============================================================

portfolio_hourly = (
    commercial.groupby(
        [
            "panel",
            "fold_id",
            "delivery_date",
            "timestamp_utc",
            "scenario_stream",
            "nomination_method",
        ],
        as_index=False,
    )
    .agg(
        actual_mwh=(
            "actual_mwh",
            "sum",
        ),
        nominated_mwh=(
            "nominated_mwh",
            "sum",
        ),
        absolute_imbalance_mwh=(
            "absolute_imbalance_mwh",
            "sum",
        ),
        incremental_eur=(
            "net_incremental_revenue_vs_perfect_eur",
            "sum",
        ),
        gross_revenue_eur=(
            "gross_revenue_eur",
            "sum",
        ),
        perfect_revenue_eur=(
            "perfect_nomination_revenue_eur",
            "sum",
        ),
    )
)


portfolio_summary_rows = []


for keys, group in portfolio_hourly.groupby(
    [
        "panel",
        "scenario_stream",
        "nomination_method",
    ],
    sort=True,
):
    (
        panel_name,
        scenario_stream,
        nomination_method,
    ) = keys

    total_actual = float(
        group[
            "actual_mwh"
        ].sum()
    )

    total_incremental = float(
        group[
            "incremental_eur"
        ].sum()
    )

    daily = (
        group.groupby(
            [
                "fold_id",
                "delivery_date",
            ],
            as_index=False,
        )[
            "incremental_eur"
        ]
        .sum()
    )

    portfolio_summary_rows.append(
        {
            "panel": panel_name,
            "scenario_stream": (
                scenario_stream
            ),
            "nomination_method": (
                nomination_method
            ),
            "hours": int(
                len(group)
            ),
            "days": int(
                group[
                    "delivery_date"
                ].nunique()
            ),
            "folds": int(
                group[
                    "fold_id"
                ].nunique()
            ),
            "total_incremental_eur": (
                total_incremental
            ),
            "incremental_eur_per_actual_mwh": (
                total_incremental
                / total_actual
                if total_actual > 0
                else np.nan
            ),
            "total_absolute_imbalance_mwh": float(
                group[
                    "absolute_imbalance_mwh"
                ].sum()
            ),
            "worst_day_incremental_eur": float(
                daily[
                    "incremental_eur"
                ].min()
            ),
            "daily_downside_cvar_5pct_eur": (
                downside_cvar(
                    daily[
                        "incremental_eur"
                    ].to_numpy(
                        dtype=float
                    ),
                    0.05,
                )
            ),
        }
    )


portfolio_summary = (
    pd.DataFrame(
        portfolio_summary_rows
    )
    .sort_values(
        [
            "panel",
            "scenario_stream",
            "total_incremental_eur",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)


display(
    portfolio_summary
)


In [ ]:
# ============================================================
# 21. FEE AND PRICE-SPREAD STRESS TESTS
# ============================================================

sensitivity_rows = []


for fee in FEE_SENSITIVITY_EUR_PER_MWH:
    for multiplier in SPREAD_STRESS_MULTIPLIERS:
        stressed = commercial.copy()

        stressed[
            "stressed_incremental_eur"
        ] = (
            multiplier
            * stressed[
                "incremental_revenue_vs_perfect_eur"
            ]
            - fee
            * stressed[
                "absolute_imbalance_mwh"
            ]
        )

        for keys, group in stressed.groupby(
            [
                "panel",
                "price_area",
                "scenario_stream",
                "nomination_method",
            ],
            sort=True,
        ):
            (
                panel_name,
                price_area,
                scenario_stream,
                nomination_method,
            ) = keys

            total_actual = float(
                group[
                    "actual_mwh"
                ].sum()
            )

            total_value = float(
                group[
                    "stressed_incremental_eur"
                ].sum()
            )

            sensitivity_rows.append(
                {
                    "additional_fee_eur_per_mwh": (
                        fee
                    ),
                    "spread_multiplier": (
                        multiplier
                    ),
                    "panel": panel_name,
                    "price_area": (
                        price_area
                    ),
                    "scenario_stream": (
                        scenario_stream
                    ),
                    "nomination_method": (
                        nomination_method
                    ),
                    "total_stressed_incremental_eur": (
                        total_value
                    ),
                    "stressed_incremental_eur_per_actual_mwh": (
                        total_value
                        / total_actual
                        if total_actual > 0
                        else np.nan
                    ),
                    "total_absolute_imbalance_mwh": float(
                        group[
                            "absolute_imbalance_mwh"
                        ].sum()
                    ),
                }
            )


commercial_sensitivity = (
    pd.DataFrame(
        sensitivity_rows
    )
)


display(
    commercial_sensitivity.loc[
        commercial_sensitivity[
            "additional_fee_eur_per_mwh"
        ].isin(
            [
                0.0,
                1.0,
                2.0,
            ]
        )
        & commercial_sensitivity[
            "spread_multiplier"
        ].eq(1.0)
    ]
    .sort_values(
        [
            "panel",
            "price_area",
            "scenario_stream",
            "additional_fee_eur_per_mwh",
            "total_stressed_incremental_eur",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
        ],
    )
)


In [ ]:
# ============================================================
# 22. COMMERCIAL SHORTLIST
# ============================================================

shortlist = commercial_summary.copy()


shortlist[
    "commercial_rank"
] = (
    shortlist.groupby(
        [
            "panel",
            "price_area",
            "scenario_stream",
        ]
    )[
        "total_incremental_revenue_vs_perfect_eur"
    ]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)


shortlist = (
    shortlist.loc[
        shortlist[
            "commercial_rank"
        ]
        <= 3
    ]
    .sort_values(
        [
            "panel",
            "price_area",
            "scenario_stream",
            "commercial_rank",
        ]
    )
)


shortlist = shortlist.merge(
    inference_vs_point[
        [
            "panel",
            "price_area",
            "scenario_stream",
            "nomination_method",
            "mean_daily_delta_eur",
            "bootstrap_ci_lower_mean_daily_eur",
            "bootstrap_ci_upper_mean_daily_eur",
            "bootstrap_probability_outperformance",
            "holm_hac_p_value",
            "holm_bootstrap_p_value",
        ]
    ],
    on=[
        "panel",
        "price_area",
        "scenario_stream",
        "nomination_method",
    ],
    how="left",
    validate="one_to_one",
)


shortlist[
    "inference_note"
] = np.where(
    shortlist[
        "nomination_method"
    ].eq(
        "point_centre"
    ),
    (
        "reference point nomination"
    ),
    np.where(
        shortlist[
            "holm_bootstrap_p_value"
        ]
        < 0.05,
        (
            "commercial difference is "
            "Holm-significant"
        ),
        (
            "commercial ranking is "
            "exploratory"
        ),
    ),
)


display(
    shortlist[
        [
            "panel",
            "price_area",
            "scenario_stream",
            "commercial_rank",
            "nomination_method",
            "total_incremental_revenue_vs_perfect_eur",
            "incremental_eur_per_actual_mwh",
            "total_absolute_imbalance_mwh",
            "hourly_downside_cvar_5pct_eur",
            "mean_daily_delta_eur",
            "bootstrap_ci_lower_mean_daily_eur",
            "bootstrap_ci_upper_mean_daily_eur",
            "inference_note",
        ]
    ]
)


In [ ]:
# ============================================================
# 23. VISUAL DIAGNOSTICS
# ============================================================

def save_current_figure(
    filename: str,
) -> None:
    path = (
        FIGURE_DIR
        / filename
    )

    plt.tight_layout()

    plt.savefig(
        path,
        dpi=160,
        bbox_inches="tight",
    )

    plt.show()
    plt.close()


for panel_name in sorted(
    commercial_summary[
        "panel"
    ].unique()
):
    for price_area in PRICE_AREAS:
        selected = (
            commercial_summary.loc[
                commercial_summary[
                    "panel"
                ].eq(panel_name)
                & commercial_summary[
                    "price_area"
                ].eq(price_area)
            ]
            .copy()
        )

        if selected.empty:
            continue

        labels = (
            selected[
                "scenario_stream"
            ]
            + "\n"
            + selected[
                "nomination_method"
            ]
        )

        ordered = (
            selected.assign(
                label=labels
            )
            .sort_values(
                "incremental_eur_per_actual_mwh"
            )
        )

        plt.figure(
            figsize=(12, 7)
        )

        plt.barh(
            ordered[
                "label"
            ],
            ordered[
                "incremental_eur_per_actual_mwh"
            ],
        )

        plt.xlabel(
            "Incremental EUR per actual MWh "
            "versus perfect nomination"
        )

        plt.ylabel(
            "Scenario stream and "
            "nomination policy"
        )

        plt.title(
            f"{panel_name.upper()} "
            f"{price_area}: "
            "commercial nomination ranking"
        )

        save_current_figure(
            (
                "notebook_04_commercial_"
                f"ranking_{panel_name}_"
                f"{price_area}.png"
            )
        )


best_context = (
    commercial_summary.sort_values(
        [
            "panel",
            "price_area",
            "scenario_stream",
            "total_incremental_revenue_vs_perfect_eur",
        ],
        ascending=[
            True,
            True,
            True,
            False,
        ],
    )
    .groupby(
        [
            "panel",
            "price_area",
            "scenario_stream",
        ],
        as_index=False,
    )
    .first()
)


for row in best_context.itertuples(
    index=False
):
    selected = commercial.loc[
        commercial[
            "panel"
        ].eq(row.panel)
        & commercial[
            "price_area"
        ].eq(row.price_area)
        & commercial[
            "scenario_stream"
        ].eq(row.scenario_stream)
        & commercial[
            "nomination_method"
        ].eq(row.nomination_method)
    ].copy()

    daily = (
        selected.groupby(
            "delivery_date",
            as_index=False,
        )[
            "net_incremental_revenue_vs_perfect_eur"
        ]
        .sum()
        .sort_values(
            "delivery_date"
        )
    )

    daily[
        "cumulative_incremental_eur"
    ] = daily[
        "net_incremental_revenue_vs_perfect_eur"
    ].cumsum()

    plt.figure(
        figsize=(12, 6)
    )

    plt.plot(
        pd.to_datetime(
            daily[
                "delivery_date"
            ]
        ),
        daily[
            "cumulative_incremental_eur"
        ],
    )

    plt.xlabel(
        "Delivery date"
    )

    plt.ylabel(
        "Cumulative incremental EUR "
        "versus perfect nomination"
    )

    plt.title(
        f"{row.panel.upper()} "
        f"{row.price_area} "
        f"{row.scenario_stream}: "
        f"{row.nomination_method}"
    )

    save_current_figure(
        (
            "notebook_04_cumulative_"
            f"{row.panel}_"
            f"{row.price_area}_"
            f"{row.scenario_stream}.png"
        )
    )


In [ ]:
# ============================================================
# 24. SAVE OUTPUTS AND FINAL VERIFICATION
# ============================================================

commercial_summary.to_csv(
    TABLE_DIR
    / "notebook_04_commercial_summary.csv",
    index=False,
)

commercial_fold_summary.to_csv(
    TABLE_DIR
    / "notebook_04_commercial_summary_by_fold.csv",
    index=False,
)

paired_vs_point.to_csv(
    TABLE_DIR
    / "notebook_04_comparison_vs_point.csv",
    index=False,
)

inference_vs_point.to_csv(
    TABLE_DIR
    / "notebook_04_inference_vs_point.csv",
    index=False,
)

raw_vs_gated_summary.to_csv(
    TABLE_DIR
    / "notebook_04_raw_vs_gated_summary.csv",
    index=False,
)

portfolio_summary.to_csv(
    TABLE_DIR
    / "notebook_04_denmark_portfolio_summary.csv",
    index=False,
)

commercial_sensitivity.to_csv(
    TABLE_DIR
    / "notebook_04_fee_and_spread_sensitivity.csv",
    index=False,
)

shortlist.to_csv(
    TABLE_DIR
    / "notebook_04_commercial_shortlist.csv",
    index=False,
)

stream_wide.to_parquet(
    SETTLEMENT_DIR
    / "notebook_04_raw_vs_gated_daily.parquet",
    index=False,
)

portfolio_hourly.to_parquet(
    SETTLEMENT_DIR
    / "notebook_04_denmark_portfolio_hourly.parquet",
    index=False,
)


blocking_failures = []


if notebook_03_verification.get(
    "verification_status"
) != "PROBABILISTIC_MODELLING_COMPLETE":
    blocking_failures.append(
        "Notebook 03 verification failed."
    )


if quarter_duplicates:
    blocking_failures.append(
        f"{quarter_duplicates} duplicate "
        "quarter-hour market keys."
    )


if not incomplete_hours.empty:
    blocking_failures.append(
        f"{len(incomplete_hours)} incomplete "
        "market hours."
    )


if missing_market_rows.any():
    blocking_failures.append(
        f"{int(missing_market_rows.sum())} "
        "forecast rows lack market prices."
    )


if signal_lookahead_violations:
    blocking_failures.append(
        f"{signal_lookahead_violations} "
        "market-signal look-ahead violations."
    )


if online_lookahead_violations:
    blocking_failures.append(
        f"{online_lookahead_violations} "
        "online-policy look-ahead violations."
    )


if (
    maximum_revenue_identity_error
    > 1e-7
):
    blocking_failures.append(
        "Commercial revenue identity failed."
    )


if support_mismatches:
    blocking_failures.append(
        f"{len(support_mismatches)} "
        "common-support mismatches."
    )


observed_contexts = set(
    commercial[
        [
            "panel",
            "price_area",
            "scenario_stream",
        ]
    ]
    .drop_duplicates()
    .itertuples(
        index=False,
        name=None,
    )
)


expected_contexts = {
    (
        panel_name,
        price_area,
        stream,
    )
    for panel_name in (
        "d2",
        "exact",
    )
    for price_area in (
        PRICE_AREAS
    )
    for stream in (
        SCENARIO_STREAMS
    )
}


if observed_contexts != expected_contexts:
    blocking_failures.append(
        "Commercial outputs do not cover "
        "all panel-area-stream contexts."
    )


verification_status = (
    "COMMERCIAL_EVALUATION_COMPLETE"
    if not blocking_failures
    else "BLOCKED"
)


def file_status(
    path: Path,
) -> dict[str, Any]:
    return {
        "path": str(
            path
        ),
        "exists": (
            path.exists()
        ),
        "size_mb": (
            round(
                path.stat().st_size
                / (1024 ** 2),
                3,
            )
            if path.exists()
            else None
        ),
    }


saved_files = [
    MARKET_DIR
    / "market_prices_quarter_hour.parquet",
    MARKET_DIR
    / "market_prices_hourly.parquet",
    NOMINATION_DIR
    / "notebook_04_nominations_hourly.parquet",
    NOMINATION_DIR
    / "candidate_quantile_daily_utility.parquet",
    SETTLEMENT_DIR
    / "notebook_04_settlement_hourly.parquet",
    TABLE_DIR
    / "notebook_04_commercial_summary.csv",
    TABLE_DIR
    / "notebook_04_commercial_summary_by_fold.csv",
    TABLE_DIR
    / "notebook_04_comparison_vs_point.csv",
    TABLE_DIR
    / "notebook_04_inference_vs_point.csv",
    TABLE_DIR
    / "notebook_04_raw_vs_gated_summary.csv",
    TABLE_DIR
    / "notebook_04_denmark_portfolio_summary.csv",
    TABLE_DIR
    / "notebook_04_fee_and_spread_sensitivity.csv",
    TABLE_DIR
    / "notebook_04_commercial_shortlist.csv",
    AUDIT_DIR
    / "market_api_request_log.csv",
    AUDIT_DIR
    / "market_price_coverage.csv",
    AUDIT_DIR
    / "online_quantile_decisions.csv",
]


verification = {
    "verification_status": (
        verification_status
    ),
    "blocking_failures": (
        blocking_failures
    ),
    "input_manifest": (
        input_manifest
    ),
    "evaluation_window": {
        "market_history_start_utc": (
            MARKET_HISTORY_START_UTC
        ),
        "evaluation_start_utc": (
            EVALUATION_START_UTC
        ),
        "evaluation_end_utc_exclusive": (
            EVALUATION_END_UTC
        ),
    },
    "commercial_specification": {
        "portfolio_share": (
            PORTFOLIO_SHARE
        ),
        "settlement_model": (
            "Nordic single imbalance price"
        ),
        "hourly_energy_to_quarter_price_assumption": (
            "net hourly deviation multiplied "
            "by mean quarter-hour spread"
        ),
        "nomination_methods": sorted(
            expected_methods
        ),
        "scenario_streams": list(
            SCENARIO_STREAMS
        ),
    },
    "strict_audit": {
        "scenario_count_per_timestamp": (
            SCENARIO_COUNT
        ),
        "quarter_key_duplicates": (
            quarter_duplicates
        ),
        "incomplete_market_hours": int(
            len(
                incomplete_hours
            )
        ),
        "missing_market_rows": int(
            missing_market_rows.sum()
        ),
        "market_signal_lookahead_violations": (
            signal_lookahead_violations
        ),
        "online_policy_lookahead_violations": (
            online_lookahead_violations
        ),
        "maximum_revenue_identity_error": (
            maximum_revenue_identity_error
        ),
        "common_support_mismatches": (
            len(
                support_mismatches
            )
        ),
    },
    "commercial_shortlist": (
        shortlist[
            [
                "panel",
                "price_area",
                "scenario_stream",
                "commercial_rank",
                "nomination_method",
                "total_incremental_revenue_vs_perfect_eur",
                "incremental_eur_per_actual_mwh",
                "total_absolute_imbalance_mwh",
                "inference_note",
            ]
        ]
        .round(6)
        .to_dict(
            orient="records"
        )
    ),
    "saved_files": [
        file_status(
            path
        )
        for path in saved_files
    ],
}


verification_path = (
    AUDIT_DIR
    / "notebook_04_verification.json"
)


with verification_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        verification,
        file,
        indent=2,
        default=str,
    )


print(
    "=== NOTEBOOK 04 VERIFICATION ==="
)

print(
    json.dumps(
        verification,
        indent=2,
        default=str,
    )
)


if blocking_failures:
    raise RuntimeError(
        "Notebook 04 verification failed: "
        f"{blocking_failures}"
    )


## Outputs

Notebook 04 writes to:

```text
outputs/notebook_04_commercial/
```

Principal artifacts:

```text
audits/notebook_04_verification.json
audits/market_api_request_log.csv
audits/market_price_coverage.csv
audits/online_quantile_decisions.csv

market/market_prices_quarter_hour.parquet
market/market_prices_hourly.parquet

nominations/notebook_04_nominations_hourly.parquet
nominations/candidate_quantile_daily_utility.parquet

settlement/notebook_04_settlement_hourly.parquet
settlement/notebook_04_raw_vs_gated_daily.parquet
settlement/notebook_04_denmark_portfolio_hourly.parquet

tables/notebook_04_commercial_summary.csv
tables/notebook_04_commercial_summary_by_fold.csv
tables/notebook_04_comparison_vs_point.csv
tables/notebook_04_inference_vs_point.csv
tables/notebook_04_raw_vs_gated_summary.csv
tables/notebook_04_denmark_portfolio_summary.csv
tables/notebook_04_fee_and_spread_sensitivity.csv
tables/notebook_04_commercial_shortlist.csv
```

Notebook 05 will use the commercial shortlist, inference tables, stress tests,
and the raw/gated stream comparison to select the final integrated decision
policy.
